# MODEL 4 — ADAPTIVE RISK + LLM/RAG + UNCERTAINTY-GATED MAPPO

This notebook is the complete sequential implementation for Model 4.

It preserves the Model-2 market/environment implementation, warms the two MAPPO actors from the validated Model-3 final checkpoint, and adds: adaptive market-risk sensitivity, RAG retrieval, a frozen local LLM policy prior, uncertainty gating, checkpointed MAPPO training, evaluation, ablations, and final artifact saving.

**Run cells strictly top-to-bottom.** Every critical stage writes a checkpoint/artifact under `/kaggle/working/gt_model4_final/`.

The research roadmap places Adaptive Risk, LLM+RAG Policy Prior, and Uncertainty-Gated LLM as consecutive stages; here they are intentionally integrated into one Model-4 system while retaining ablations so each contribution can still be measured independently.


## 0 — Model-4 runtime, reproducibility, and checkpoint layout

In [ ]:
# ============================================================
# MODEL 4 — GLOBAL REPRODUCIBILITY / PATH SETUP
# ============================================================
import os, sys, math, json, glob, time, re, hashlib, random, subprocess, shutil
from pathlib import Path

M4_ROOT = Path("/kaggle/working/gt_model4_final")
M4_CKPT = M4_ROOT / "checkpoints"
M4_ART = M4_ROOT / "artifacts"
M4_LOG = M4_ROOT / "logs"
for p in (M4_ROOT, M4_CKPT, M4_ART, M4_LOG):
    p.mkdir(parents=True, exist_ok=True)

M4_SEED = int(os.environ.get("FDM_M4_SEED", "700"))
M4_CONTROL_UPDATES = int(os.environ.get("FDM_M4_CONTROL_UPDATES", "1000"))
M4_FULL_UPDATES = int(os.environ.get("FDM_M4_FULL_UPDATES", "1500"))
M4_ABLATION_UPDATES = int(os.environ.get("FDM_M4_ABLATION_UPDATES", "1000"))
M4_ROLLOUT = int(os.environ.get("FDM_M4_ROLLOUT", "25"))
M4_CKPT_INTERVAL = int(os.environ.get("FDM_M4_CKPT_INTERVAL", "50"))
M4_EVAL_EPISODES = int(os.environ.get("FDM_M4_EVAL_EPISODES", "12"))

# Model-3 risk settings carried forward as the starting scientific reference.
M4_BASE_RISK_COEF = 0.25
M4_RISK_SCALE = 1.50
M4_CVAR_LIMIT = 4.10
M4_LAMBDA_INIT = 0.0
M4_LAMBDA_LR = 0.01
M4_CVAR_ALPHA = 0.10
M4_NUM_QUANTILES = 25

# PPO settings kept close to the validated project trainer.
M4_LR = 1.0e-4
M4_GAMMA = 0.97
M4_GAE_LAM = 0.95
M4_CLIP = 0.20
M4_EPOCHS = 4
M4_ENT_COEF = 0.015
M4_VALUE_COEF = 0.50
M4_RISK_VALUE_COEF = 0.25
M4_MAX_GRAD_NORM = 0.50

# LLM/RAG policy-prior settings.
M4_LLM_ID = os.environ.get("FDM_M4_LLM_ID", "google/flan-t5-small")
M4_LLM_MAX_TOKENS = 48
M4_LLM_REFRESH_UPDATES = int(os.environ.get("FDM_M4_LLM_REFRESH_UPDATES", "4"))
M4_LLM_PRIOR_COEF = 0.20
M4_KL_COEF = 0.03
M4_MIN_GATE = 0.05

# Fixed/stable seed setup.
os.environ["PYTHONHASHSEED"] = str(M4_SEED)
random.seed(M4_SEED)

import numpy as np
import pandas as pd

np.random.seed(M4_SEED)

print("MODEL 4 runtime ready")
print(" seed                 :", M4_SEED)
print(" control updates      :", M4_CONTROL_UPDATES)
print(" full updates         :", M4_FULL_UPDATES)
print(" rollout length       :", M4_ROLLOUT)
print(" checkpoint interval  :", M4_CKPT_INTERVAL)
print(" root                 :", M4_ROOT)


## 1 — Rebuild the validated Model-2 market/environment definitions

These cells are copied from the checkpointed project notebook so Model 4 uses the same market, observation space, action space, calibration, and metrics. No Model-2 training is executed here.

In [ ]:
# ===== Setup =====
import os, sys, math, json, glob, time, subprocess, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- runtime knobs (env-var overridable so you can do quick smoke runs) ---
FDM_EPISODES     = int(os.environ.get("FDM_EPISODES", "6000"))    # MAPPO training episodes
FDM_PPO_EPISODES = int(os.environ.get("FDM_PPO_EPISODES", "6000")) # PPO baseline episodes
FDM_EVAL_EPS     = int(os.environ.get("FDM_EVAL_EPS", "12"))      # eval episodes per policy config
FDM_SEED         = int(os.environ.get("FDM_SEED", "0"))
# Model-2 (Stages 5-7) knobs
FDM_FC_WEEKS     = int(os.environ.get("FDM_FC_WEEKS", "200"))     # simulated weeks for forecasting data
FDM_WM_EPISODES  = int(os.environ.get("FDM_WM_EPISODES", "100"))  # episodes for world-model data
FDM_CMP_EPISODES = int(os.environ.get("FDM_CMP_EPISODES", "4000"))# forecast vs no-forecast comparison runs
FDM_CF_EPISODES  = int(os.environ.get("FDM_CF_EPISODES", "2000")) # counterfactual-advantage runs

def _ensure(pkg, extra=None):
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        cmd = [sys.executable, "-m", "pip", "install", "-q", pkg] + (extra or [])
        subprocess.run(cmd, check=True)
        __import__(pkg)

_ensure("kagglehub")
_ensure("torch")
import torch
import kagglehub

torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("numpy", np.__version__, "| pandas", pd.__version__, "| torch", torch.__version__, "| device:", DEVICE)
print("run sizes -> MAPPO episodes:", FDM_EPISODES, "| PPO episodes:", FDM_PPO_EPISODES, "| eval eps:", FDM_EVAL_EPS)

In [ ]:
# ===== Dataset 1: Delivery Center Brazil (order-level food/goods delivery) =====
DC_OK = False
DC_DEFAULTS = dict(base_delay_min=42.0, delay_noise_std=8.0, fee_ratio=0.111,
                   base_lambda=60.0, n_couriers=70, orders_per_courier_hour=1.3)
try:
    dc_dir = kagglehub.dataset_download("nosbielcs/brazilian-delivery-center")
    dcg = lambda name: glob.glob(os.path.join(dc_dir, name))[0]
    stores = pd.read_csv(dcg("stores.csv"), encoding="latin-1")
    orders = pd.read_csv(dcg("orders.csv"), usecols=[
        "order_id", "store_id", "order_status", "order_amount", "order_delivery_fee",
        "order_created_hour", "order_created_day", "order_created_month", "order_created_year",
        "order_moment_created", "order_moment_delivered", "order_metric_cycle_time"])
    deliveries = pd.read_csv(dcg("deliveries.csv"),
                             usecols=["delivery_order_id", "driver_id", "delivery_status"])
    # keep FOOD-segment stores only (this is a food-delivery study)
    food_stores = stores.loc[stores.store_segment == "FOOD", ["store_id", "hub_id"]]
    orders = orders.merge(food_stores, on="store_id", how="inner")
    fin = orders[orders.order_status == "FINISHED"].copy()
    ts = pd.to_datetime(fin.order_moment_created, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    fin["dow"] = ts.dt.dayofweek
    fin = fin.dropna(subset=["dow"])
    fin["dow"] = fin["dow"].astype(int)
    print("Delivery Center: %d FOOD orders finished, %d hubs" % (len(fin), food_stores.hub_id.nunique()))
    DC_OK = True
except Exception as e:
    print("Delivery Center download failed (%s: %s) -> using documented defaults" % (type(e).__name__, e))
    

In [ ]:
# ===== Dataset 2: Zomato Bangalore (market price structure) =====
ZOM_OK = False
ZOM_DEFAULTS = dict(basket=400.0, online_share=0.82, demand_gini=0.60, rating_mean=3.6)
try:
    zom_dir = kagglehub.dataset_download("himanshupoddar/zomato-bangalore-restaurants")
    zom = pd.read_csv(glob.glob(os.path.join(zom_dir, "*.csv"))[0],
                      usecols=["name", "rate", "votes", "approx_cost(for two people)", "online_order"])
    cost = (zom["approx_cost(for two people)"].astype(str).str.replace(",", "", regex=False)
            .pipe(pd.to_numeric, errors="coerce"))
    zom = zom.assign(cost=cost)
    zom = zom.dropna(subset=["cost", "votes"])
    zom = zom[(zom.cost >= 40) & (zom.cost <= 3000)]
    ZOM_OK = True
    print("Zomato: %d restaurants | online-order share %.2f" % (len(zom), (zom.online_order == "Yes").mean()))
except Exception as e:
    print("Zomato download failed (%s: %s) -> using documented defaults" % (type(e).__name__, e))

In [ ]:
# ===== Calibration A: demand profile (hour of day, day of week) =====
hour_profile = None
day_profile = None

if DC_OK:
    # hour-of-day and day-of-week multipliers, normalized to mean 1.0.
    # The raw empirical hour profile spans ~200x (late-night peaks vs morning);
    # we temper it (power 0.6) so a fixed-courier zone can plausibly serve it.
    by_hour = fin.groupby("order_created_hour").size().reindex(range(24), fill_value=0)
    by_dow  = fin.groupby("dow").size().reindex(range(7), fill_value=0)
    hour_profile = (by_hour / by_hour.mean()).to_numpy() ** 0.6
    hour_profile /= hour_profile.mean()
    day_profile  = (by_dow  / by_dow.mean()).to_numpy()
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].bar(range(24), hour_profile, color="#4C72B0")
    ax[0].set_title("Demand multiplier by hour (Delivery Center FOOD orders)")
    ax[0].set_xlabel("hour of day"); ax[0].grid(alpha=.3)
    ax[1].bar(range(7), day_profile, color="#DD8452")
    ax[1].set_title("Demand multiplier by weekday"); ax[1].set_xlabel("Mon .. Sun")
    ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("peak hours:", np.argsort(hour_profile)[::-1][:3], "| weakest:", np.argsort(hour_profile)[:3])
else:
    # documented fallback: lunch + dinner peaks (industry-standard shape)
    hour_profile = np.array([.45,.30,.22,.18,.18,.22,.35,.60,.85,.90,.85,1.0,
                             1.5,1.7,1.3,1.0,.95,1.05,1.35,1.7,1.85,1.6,1.1,.7])
    hour_profile /= hour_profile.mean()
    day_profile = np.array([.95,.90,.92,.98,1.08,1.15,1.12])
    print("using fallback demand profile (assumed)")

SIGMA_LAMBDA = 0.12   # multiplicative demand noise — assumption (swept in robustness stage)
print("sigma_lambda =", SIGMA_LAMBDA, "(assumed)")

In [ ]:
# ===== Calibration B: delivery operations + market scale (per-hub zone) =====
calib_ops = {}
if DC_OK:
    # --- delivery cycle time (created -> delivered), robust window 5..120 min
    ct = pd.to_numeric(fin.order_metric_cycle_time, errors="coerce")
    ct = ct[(ct >= 5) & (ct <= 120)]
    calib_ops["base_delay_min"] = float(ct.median())
    # within-hour variation only; the day-level spread is modeled via the load term
    calib_ops["delay_noise_std"] = 6.0
    # severe-delay threshold: problem statement allows "45 min OR > 90th percentile of
    # the calibration distribution" -> with a ~41-min data median we take the p90 route
    calib_ops["sla_min"] = float(max(45.0, ct.quantile(0.90)))

    # --- delivery-fee-to-order-value ratio
    m = fin[(fin.order_delivery_fee > 0) & (fin.order_delivery_fee < fin.order_amount)]
    calib_ops["fee_ratio"] = float((m.order_delivery_fee / m.order_amount).median())

    # --- market scale: demand-anchored staffing of one representative busy hub
    hh = fin.groupby(["hub_id", "order_created_year", "order_created_month",
                      "order_created_day", "order_created_hour"]).size()
    hub_med = hh.groupby("hub_id").median()
    pct75 = hub_med.quantile(.75)
    busy_hubs = hub_med[hub_med >= pct75].index
    chosen = int(hub_med.loc[busy_hubs].idxmax()) if len(busy_hubs) else int(hub_med.idxmax())
    calib_ops["base_lambda"] = float(hub_med.loc[chosen])
    # staff the zone so that at typical participation (0.75) and 1.3 orders/hour/courier
    # the mean load is ~0.75 (scarcity binds only at the calibrated demand peaks)
    calib_ops["n_couriers"] = int(np.clip(round(calib_ops["base_lambda"] / (0.75 * 1.3 * 0.75)), 20, 400))

    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].hist(ct, bins=60, color="#4C72B0", alpha=.85)
    ax[0].axvline(calib_ops["base_delay_min"], color="crimson", ls="--",
                  label="median %.1f min" % calib_ops["base_delay_min"])
    ax[0].axvline(calib_ops["sla_min"], color="black", ls=":",
                  label="SLA p90 %.1f min" % calib_ops["sla_min"])
    ax[0].set_title("Delivery cycle time (min), FOOD orders"); ax[0].legend(); ax[0].grid(alpha=.3)
    hub_med.sort_values(ascending=False).head(15).plot(kind="bar", ax=ax[1], color="#55A868")
    ax[1].axhline(calib_ops["base_lambda"], color="crimson", ls="--", label="chosen hub")
    ax[1].set_title("Median hourly orders per hub (top 15)"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    calib_ops = {k: DC_DEFAULTS[k] for k in ("base_delay_min", "delay_noise_std", "fee_ratio",
                                             "base_lambda", "n_couriers")}
    calib_ops["sla_min"] = 45.0
print(calib_ops)

In [ ]:
# ===== Calibration C: money structure from Zomato + summary table =====
calib_money = {}
if ZOM_OK:
    calib_money["basket"] = float(zom.cost.median())          # order value in INR (meal for two as one order)
    calib_money["online_share"] = float((zom.online_order == "Yes").mean())
    votes_online = zom.loc[zom.online_order == "Yes", "votes"].to_numpy(dtype=float)
    v = np.sort(votes_online); n = v.size
    calib_money["demand_gini"] = float((2 * (np.arange(1, n + 1) * v).sum() - (n + 1) * v.sum()) / (n * v.sum()))
    calib_money["rating_mean"] = float(pd.to_numeric(zom.rate.astype(str).str.split("/").str[0],
                                                     errors="coerce").mean())
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].hist(zom.cost.clip(0, 1500), bins=60, color="#4C72B0", alpha=.85)
    ax[0].axvline(calib_money["basket"], color="crimson", ls="--",
                  label="median basket = %d INR" % calib_money["basket"])
    ax[0].set_title("Order value (approx cost for two, INR)"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].hist(np.log10(zom.votes.clip(lower=1)), bins=60, color="#DD8452", alpha=.85)
    ax[1].set_title("Popularity (log10 votes) -> demand concentration Gini=%.2f" % calib_money["demand_gini"])
    ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    calib_money = dict(ZOM_DEFAULTS)

# ---- assemble the calibrated parameters ----
basket = calib_money["basket"]
fee_ref = calib_ops["fee_ratio"] * basket          # fee scales with order value (fee_ratio from data)
# operations overhead ≈ 25% of one platform's commission gross (assumption; a platform
# captures ~40% of zone orders, so its fixed cost must scale with its own volume)
fixed_cost = 0.25 * calib_ops["base_lambda"] * 0.22 * basket * 0.40
patience_mean = max(60.0, calib_ops["base_delay_min"] + 40.0)   # cancel patience vs realized delay
patience_std = max(12.0, 2.0 * calib_ops["delay_noise_std"])

rows = [
    ("base_lambda (orders/hour in zone)", round(calib_ops["base_lambda"], 1), "Delivery Center" if DC_OK else "assumed"),
    ("n_couriers in zone (demand-anchored)", calib_ops["n_couriers"], "derived (data)"),
    ("orders_per_courier_hour", 1.3, "assumption (gig-work norm)"),
    ("base_delay_min (median cycle)", round(calib_ops["base_delay_min"], 1), "Delivery Center" if DC_OK else "assumed"),
    ("delay_noise_std (min)", calib_ops["delay_noise_std"], "assumption (within-hour)"),
    ("SLA threshold (min)", round(calib_ops["sla_min"], 1), "PS rule: max(45, data p90)"),
    ("cancel patience (min)", round(patience_mean, 1), "derived vs base delay"),
    ("basket (INR/order)", basket, "Zomato" if ZOM_OK else "assumed"),
    ("delivery_fee_ref (INR)", round(fee_ref, 1), "derived (fee ratio x basket)"),
    ("fixed ops cost/hour (per platform)", round(fixed_cost, 0), "derived (25% of platform commission)"),
    ("demand concentration Gini", round(calib_money.get("demand_gini", .6), 2), "Zomato" if ZOM_OK else "assumed"),
    ("commission_rate", 0.22, "assumption"),
    ("courier_pay_ref (INR/order)", 35.0, "assumption"),
    ("incentive_ref (INR/order)", 10.0, "assumption"),
    ("share_A / share_B / multi-homing", "0.45 / 0.35 / 0.20", "assumption"),
]
calib_table = pd.DataFrame(rows, columns=["parameter", "value", "source"])
print(calib_table.to_string(index=False))

In [ ]:
# ===== Stage 1: the analytical market simulator =====
from dataclasses import dataclass, field

PRICE_GRID = (0.8, 0.9, 1.0, 1.1, 1.2)
INCENTIVE_GRID = (0.0, 0.5, 1.0, 1.5, 2.0)
N_ACTIONS = len(PRICE_GRID) * len(INCENTIVE_GRID)
UNFULFILLED_WAIT_MIN, MAX_DELAY_MIN = 90.0, 120.0

def normal_sf(x, mu, sigma):
    return 0.5 * math.erfc((x - mu) / (sigma * math.sqrt(2.0)))

@dataclass
class MarketConfig:
    seed: int = FDM_SEED
    T: int = 168
    base_lambda: float = 60.0
    sigma_lambda: float = 0.12
    hour_profile: tuple = tuple(hour_profile)
    day_profile: tuple = tuple(day_profile)
    rain_start_p: float = 0.15
    rain_stay_p: float = 0.60
    rain_demand_mult: float = 1.20
    rain_delay_mult: float = 1.15
    beta_price: float = -2.2
    beta_incentive: float = 0.45
    beta_wait: float = -1.6
    brand_A: float = 0.15
    brand_B: float = 0.0
    wait_ref: float = 30.0
    n_couriers: int = 60
    share_A: float = 0.45
    share_B: float = 0.35
    base_participation: float = 0.75
    kappa_incentive: float = 0.15
    kappa_earn: float = 0.50
    earn_ref: float = 55.0
    orders_per_courier_hour: float = 1.3
    courier_skill_cv: float = 0.40
    ewma_alpha: float = 0.3
    ewma_wait_alpha: float = 0.4
    base_delay_min: float = 42.0
    delay_load_coef: float = 0.3
    delay_noise_std: float = 6.0
    sla_min: float = 45.0
    patience_mean: float = 60.0
    patience_std: float = 12.0
    switching_share: float = 0.35
    enable_courier_switching: bool = True
    affiliation_switch_p: float = 0.15
    basket: float = 400.0
    commission_rate: float = 0.22
    delivery_fee_ref: float = 44.0
    courier_pay_ref: float = 35.0
    incentive_ref: float = 10.0
    fixed_cost_per_step: float = 1500.0
    c_delay: float = 0.25
    reward_scale: float = 1000.0
    c_f: float = 10.0
    c_s: float = 10.0
    c_w: float = 10.0
    spend_headroom: float = 1.5
    cvar_beta: float = 0.95
    demand_ewma_alpha: float = 0.2
    obs_noise_std: float = 0.10

    @property
    def mean_lambda(self):
        return self.base_lambda

def _logit_shares(cfg, pa, wa, pb, wb, wait_a, wait_b):
    ua = cfg.brand_A + cfg.beta_price * (pa - 1.0) + cfg.beta_incentive * wa + cfg.beta_wait * (wait_a / cfg.wait_ref - 1.0)
    ub = cfg.brand_B + cfg.beta_price * (pb - 1.0) + cfg.beta_incentive * wb + cfg.beta_wait * (wait_b / cfg.wait_ref - 1.0)
    m = max(ua, ub, 0.0)
    ea, eb, e0 = math.exp(ua - m), math.exp(ub - m), math.exp(-m)
    z = ea + eb + e0
    return ea / z, eb / z, e0 / z

def _participation(cfg, w, ewma_earn):
    gap = ewma_earn / cfg.earn_ref - 1.0
    return float(np.clip(cfg.base_participation + cfg.kappa_incentive * (w - 1.0) + cfg.kappa_earn * gap, 0.10, 1.0))

def analytical_core(cfg, demand, rain, pa, wa, pb, wb, wait_a, wait_b, earn_a, earn_b):
    """Deterministic expectation of one market step (no RNG, no side effects)."""
    sa, sb, _ = _logit_shares(cfg, pa, wa, pb, wb, wait_a, wait_b)
    na, nb = demand * sa, demand * sb
    part_a = _participation(cfg, wa, earn_a)
    part_b = _participation(cfg, wb, earn_b)
    n_multi_active = cfg.n_couriers * (1.0 - cfg.share_A - cfg.share_B) * 0.5 * (part_a + part_b)
    if cfg.enable_courier_switching:
        aa = wa + cfg.kappa_earn * (earn_a / cfg.earn_ref - 1.0)
        ab = wb + cfg.kappa_earn * (earn_b / cfg.earn_ref - 1.0)
        m = max(aa, ab)
        wta = math.exp(aa - m) / (math.exp(aa - m) + math.exp(ab - m))
    else:
        wta = 0.5
    active_a = cfg.n_couriers * cfg.share_A * part_a + n_multi_active * wta
    active_b = cfg.n_couriers * cfg.share_B * part_b + n_multi_active * (1.0 - wta)
    cap_a, cap_b = active_a * cfg.orders_per_courier_hour, active_b * cfg.orders_per_courier_hour
    ful_a, un_a = min(na, cap_a), max(na - cap_a, 0.0)
    ful_b, un_b = min(nb, cap_b), max(nb - cap_b, 0.0)
    if cfg.enable_courier_switching:
        moved_ab = min(un_a * cfg.switching_share, max(cap_b - ful_b, 0.0))
        moved_ba = min(un_b * cfg.switching_share, max(cap_a - ful_a, 0.0))
        ful_a += moved_ba; ful_b += moved_ab; un_a -= moved_ab; un_b -= moved_ba
    delay = {}
    for name, n, cap in (("a", na, cap_a), ("b", nb, cap_b)):
        d = cfg.base_delay_min * (1.0 + cfg.delay_load_coef * (n / max(cap, 1e-9)))
        if rain:
            d *= cfg.rain_delay_mult
        d = min(d, MAX_DELAY_MIN)
        sig = math.sqrt(cfg.delay_noise_std ** 2 + cfg.patience_std ** 2)
        delay[name] = dict(delay=d, sla_rate=normal_sf(cfg.sla_min, d, cfg.delay_noise_std),
                           cancel_rate=normal_sf(cfg.patience_mean, d, sig))
    eff_a = max(ful_a * (1.0 - delay["a"]["cancel_rate"]), 0.0)
    eff_b = max(ful_b * (1.0 - delay["b"]["cancel_rate"]), 0.0)
    econ = {}
    for name, p, w, eff in (("a", pa, wa, eff_a), ("b", pb, wb, eff_b)):
        rev = (cfg.commission_rate * cfg.basket + cfg.delivery_fee_ref * p) * eff
        pay = (cfg.courier_pay_ref + cfg.incentive_ref * w) * eff
        econ[name] = dict(revenue=rev, payout=pay, incentive_spend=cfg.incentive_ref * w * eff,
                          profit=rev - pay - cfg.fixed_cost_per_step,
                          routine_delay_cost=cfg.c_delay * eff * min(delay[name]["delay"], cfg.sla_min))
    d_eff_a = (eff_a * delay["a"]["delay"] + un_a * UNFULFILLED_WAIT_MIN) / max(na, 1e-9)
    d_eff_b = (eff_b * delay["b"]["delay"] + un_b * UNFULFILLED_WAIT_MIN) / max(nb, 1e-9)
    out = dict(shares=(sa, sb, 1.0 - sa - sb), n_a=na, n_b=nb, cap_a=cap_a, cap_b=cap_b,
               active_a=active_a, active_b=active_b, ful_a=ful_a, ful_b=ful_b,
               un_a=un_a, un_b=un_b, eff_a=eff_a, eff_b=eff_b,
               delay_a=delay["a"]["delay"], delay_b=delay["b"]["delay"],
               sla_a=delay["a"]["sla_rate"], sla_b=delay["b"]["sla_rate"],
               d_eff_a=d_eff_a, d_eff_b=d_eff_b)
    for name in ("a", "b"):
        for k, v in econ[name].items():
            out[k + "_" + name] = v
    return out

@dataclass
class CourierState:
    primary: np.ndarray
    count_a: np.ndarray
    count_b: np.ndarray
    z: np.ndarray
    earnings: np.ndarray
    served_a: np.ndarray
    served_b: np.ndarray
    skill: np.ndarray

@dataclass
class StepRecord:
    t: int; hour: int; dow: int; rain: bool; lam: float; demand: int
    p_a: float; w_a: float; p_b: float; w_b: float
    n_a: float; n_b: float; cap_a: float; cap_b: float
    fulfilled_a: float; fulfilled_b: float; unfulfilled_a: float; unfulfilled_b: float
    cancel_a: float; cancel_b: float; delay_a: float; delay_b: float
    sla_a: float; sla_b: float; profit_a: float; profit_b: float
    revenue_a: float; revenue_b: float
    incentive_spend_a: float; incentive_spend_b: float
    routine_delay_cost_a: float; routine_delay_cost_b: float

class Simulator:
    def __init__(self, cfg):
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        self.reset()

    def reset(self):
        cfg, n = self.cfg, self.cfg.n_couriers
        self.rng = np.random.default_rng(cfg.seed)
        na = int(round(n * cfg.share_A)); nb = int(round(n * cfg.share_B))
        primary = np.full(n, 2, dtype=np.int8); primary[:na] = 0; primary[na:na + nb] = 1
        sig = cfg.courier_skill_cv
        skill = self.rng.lognormal(-0.5 * sig ** 2, sig, n)
        self.st = dict(t=0, rain=False,
                       ewma_wait_a=cfg.base_delay_min, ewma_wait_b=cfg.base_delay_min,
                       ewma_earn_a=cfg.earn_ref, ewma_earn_b=cfg.earn_ref,
                       ewma_demand=cfg.mean_lambda,
                       ewma_ful_a=1.0, ewma_ful_b=1.0, ewma_can_a=0.0, ewma_can_b=0.0)
        self.cs = CourierState(primary=primary,
                               count_a=(primary == 0).astype(float), count_b=(primary == 1).astype(float),
                               z=np.where(primary == 0, 0, 1).astype(np.int8),
                               earnings=np.zeros(n), served_a=np.zeros(n, bool), served_b=np.zeros(n, bool),
                               skill=skill)
        self.records = []

    def snapshot_noise(self):
        return {"bg": self.rng.bit_generator.state}

    def restore_noise(self, snap):
        self.rng.bit_generator.state = snap["bg"]

    def step(self, a_a, a_b):
        cfg, st = self.cfg, self.st
        pa, wa = PRICE_GRID[int(a_a[0])], INCENTIVE_GRID[int(a_a[1])]
        pb, wb = PRICE_GRID[int(a_b[0])], INCENTIVE_GRID[int(a_b[1])]
        t, hour, dow = st["t"], st["t"] % 24, st["t"] // 24
        if hour == 0:
            st["rain"] = bool(self.rng.random() < cfg.rain_start_p)
        else:
            st["rain"] = bool(self.rng.random() < (cfg.rain_stay_p if st["rain"] else cfg.rain_start_p))
        lam = (cfg.base_lambda * cfg.hour_profile[hour] * cfg.day_profile[dow]
               * (cfg.rain_demand_mult if st["rain"] else 1.0)
               * (1.0 + self.rng.normal(0.0, cfg.sigma_lambda)))
        demand = int(self.rng.poisson(max(lam, 0.0)))
        out = analytical_core(cfg, demand, st["rain"], pa, wa, pb, wb,
                              st["ewma_wait_a"], st["ewma_wait_b"], st["ewma_earn_a"], st["ewma_earn_b"])
        # action-invariant noise stream (CRN-ready): one uniform per customer
        u = self.rng.random(demand) if demand else np.empty(0)
        s_a, s_b = out["shares"][0], out["shares"][1]
        n_a = float((u < s_a).sum())
        n_b = float(((u >= s_a) & (u < s_a + s_b)).sum())
        ful_a, ful_b = min(n_a, out["cap_a"]), min(n_b, out["cap_b"])
        un_a, un_b = n_a - ful_a, n_b - ful_b
        if cfg.enable_courier_switching:
            moved_ab = min(un_a * cfg.switching_share, max(out["cap_b"] - ful_b, 0.0))
            moved_ba = min(un_b * cfg.switching_share, max(out["cap_a"] - ful_a, 0.0))
            ful_a += moved_ba; ful_b += moved_ab; un_a -= moved_ab; un_b -= moved_ba
        sig_c = math.sqrt(cfg.delay_noise_std ** 2 + cfg.patience_std ** 2)
        can_a = ful_a * normal_sf(cfg.patience_mean, out["delay_a"], sig_c)
        can_b = ful_b * normal_sf(cfg.patience_mean, out["delay_b"], sig_c)
        eff_a, eff_b = max(ful_a - can_a, 0.0), max(ful_b - can_b, 0.0)
        rev_a = (cfg.commission_rate * cfg.basket + cfg.delivery_fee_ref * pa) * eff_a
        rev_b = (cfg.commission_rate * cfg.basket + cfg.delivery_fee_ref * pb) * eff_b
        spend_a = cfg.incentive_ref * wa * eff_a
        spend_b = cfg.incentive_ref * wb * eff_b
        rdc_a = cfg.c_delay * eff_a * min(out["delay_a"], cfg.sla_min)
        rdc_b = cfg.c_delay * eff_b * min(out["delay_b"], cfg.sla_min)
        prof_a = rev_a - (cfg.courier_pay_ref * eff_a + spend_a) - cfg.fixed_cost_per_step
        prof_b = rev_b - (cfg.courier_pay_ref * eff_b + spend_b) - cfg.fixed_cost_per_step
        rec = StepRecord(t=t, hour=hour, dow=dow, rain=st["rain"], lam=lam, demand=demand,
                         p_a=pa, w_a=wa, p_b=pb, w_b=wb, n_a=n_a, n_b=n_b,
                         cap_a=out["cap_a"], cap_b=out["cap_b"],
                         fulfilled_a=ful_a, fulfilled_b=ful_b, unfulfilled_a=un_a, unfulfilled_b=un_b,
                         cancel_a=can_a, cancel_b=can_b, delay_a=out["delay_a"], delay_b=out["delay_b"],
                         sla_a=out["sla_a"], sla_b=out["sla_b"], profit_a=prof_a, profit_b=prof_b,
                         revenue_a=rev_a, revenue_b=rev_b,
                         incentive_spend_a=spend_a, incentive_spend_b=spend_b,
                         routine_delay_cost_a=rdc_a, routine_delay_cost_b=rdc_b)
        self.records.append(rec)
        self._assign_couriers(rec, out)
        self._update_state(rec, out)
        st["t"] += 1
        return rec

    def _assign_couriers(self, rec, out):
        cfg, cs, st = self.cfg, self.cs, self.st
        cs.served_a[:] = False; cs.served_b[:] = False
        n = cfg.n_couriers
        u_part, u_choice, u_rank = self.rng.random(n), self.rng.random(n), self.rng.random(n)
        part_a = _participation(cfg, rec.w_a, st["ewma_earn_a"])
        part_b = _participation(cfg, rec.w_b, st["ewma_earn_b"])
        if cfg.enable_courier_switching:
            aa = rec.w_a + cfg.kappa_earn * (st["ewma_earn_a"] / cfg.earn_ref - 1.0)
            ab = rec.w_b + cfg.kappa_earn * (st["ewma_earn_b"] / cfg.earn_ref - 1.0)
            m = max(aa, ab)
            wta = math.exp(aa - m) / (math.exp(aa - m) + math.exp(ab - m))
        else:
            wta = 0.5
        is_a, is_b, is_m = cs.primary == 0, cs.primary == 1, cs.primary == 2
        part_vec = np.where(is_a, part_a, np.where(is_b, part_b, 0.5 * (part_a + part_b)))
        active = u_part < part_vec
        m_to_a = active & is_m & (u_choice < wta)
        act_a = (is_a & active) | m_to_a
        act_b = (is_b & active) | (active & is_m & ~m_to_a)
        eff_a = max(rec.fulfilled_a - rec.cancel_a, 0.0)
        eff_b = max(rec.fulfilled_b - rec.cancel_b, 0.0)
        self._distribute(act_a, u_rank, eff_a, cfg.courier_pay_ref + cfg.incentive_ref * rec.w_a, cs.served_a)
        self._distribute(act_b, u_rank, eff_b, cfg.courier_pay_ref + cfg.incentive_ref * rec.w_b, cs.served_b)
        st["real_earn_a"] = ((rec.incentive_spend_a + cfg.courier_pay_ref * eff_a) / max(act_a.sum(), 1)) if act_a.sum() else cfg.earn_ref
        st["real_earn_b"] = ((rec.incentive_spend_b + cfg.courier_pay_ref * eff_b) / max(act_b.sum(), 1)) if act_b.sum() else cfg.earn_ref

    def _distribute(self, active, u_rank, eff_orders, per_order_pay, flag):
        cfg, cs = self.cfg, self.cs
        idx = np.flatnonzero(active)
        if idx.size == 0 or eff_orders < 1.0:
            return
        flag[idx] = True
        skill = np.maximum(cs.skill[idx], 1e-6)
        raw = eff_orders * skill / skill.sum()
        counts = np.floor(raw)
        resid = int(round(eff_orders - counts.sum()))
        if resid:
            tie = (raw - counts) + 1e-6 * u_rank[idx]
            counts[np.argsort(tie)[::-1][:resid]] += 1.0
        cs.earnings[idx] += counts * per_order_pay

    def _update_state(self, rec, out):
        cfg, st, cs = self.cfg, self.st, self.cs
        a = cfg.ewma_alpha
        st["ewma_earn_a"] = (1 - a) * st["ewma_earn_a"] + a * st["real_earn_a"]
        st["ewma_earn_b"] = (1 - a) * st["ewma_earn_b"] + a * st["real_earn_b"]
        aw = cfg.ewma_wait_alpha
        st["ewma_wait_a"] = (1 - aw) * st["ewma_wait_a"] + aw * out["d_eff_a"]
        st["ewma_wait_b"] = (1 - aw) * st["ewma_wait_b"] + aw * out["d_eff_b"]
        ad = cfg.demand_ewma_alpha
        st["ewma_demand"] = (1 - ad) * st["ewma_demand"] + ad * rec.demand
        or_a = rec.fulfilled_a + rec.unfulfilled_a
        or_b = rec.fulfilled_b + rec.unfulfilled_b
        st["ewma_ful_a"] = (1 - ad) * st["ewma_ful_a"] + ad * rec.fulfilled_a / max(or_a, 1e-9)
        st["ewma_ful_b"] = (1 - ad) * st["ewma_ful_b"] + ad * rec.fulfilled_b / max(or_b, 1e-9)
        st["ewma_can_a"] = (1 - ad) * st["ewma_can_a"] + ad * rec.cancel_a / max(rec.fulfilled_a, 1e-9)
        st["ewma_can_b"] = (1 - ad) * st["ewma_can_b"] + ad * rec.cancel_b / max(rec.fulfilled_b, 1e-9)
        decay = 1.0 - 1.0 / 24.0
        cs.count_a *= decay; cs.count_b *= decay
        cs.count_a += cs.served_a; cs.count_b += cs.served_b
        prev_z = cs.z.copy()
        cs.z = np.where(cs.count_a > cs.count_b, 0, np.where(cs.count_b > cs.count_a, 1, prev_z)).astype(np.int8)
        if cfg.enable_courier_switching and rec.t % 24 == 23:
            gap_a = max(0.0, (st["ewma_earn_b"] - st["ewma_earn_a"]) / cfg.earn_ref)
            gap_b = max(0.0, (st["ewma_earn_a"] - st["ewma_earn_b"]) / cfg.earn_ref)
            sw_ab = (cs.primary == 0) & (self.rng.random(cs.primary.size) < min(cfg.affiliation_switch_p * gap_a, 0.5))
            sw_ba = (cs.primary == 1) & (self.rng.random(cs.primary.size) < min(cfg.affiliation_switch_p * gap_b, 0.5))
            cs.primary = np.where(sw_ab, 1, np.where(sw_ba, 0, cs.primary))

print("Simulator classes defined.")

In [ ]:
# ===== Stage 2: two-agent RL environment + baseline policies =====
from collections import deque

OBS_DIM = 14  # base observation; +1 when a forecast_fn is attached

def action_to_pair(a):
    return divmod(int(a), len(INCENTIVE_GRID))

def pair_to_action(pair):
    return int(pair[0]) * len(INCENTIVE_GRID) + int(pair[1])

class FoodDeliveryEnv:
    def __init__(self, cfg, forecast_fn=None):
        self.cfg = cfg
        self.sim = Simulator(cfg)
        # optional demand forecaster: callable(hist, hour_next, dow_next) -> D_hat / mean_lambda
        self.forecast_fn = forecast_fn
        self._dhist = deque(maxlen=48)
        self._fc = 1.0
        self._last = {"A": (1.0, 1.0), "B": (1.0, 1.0)}

    def reset(self, seed=None):
        if seed is not None:
            self.cfg.seed = seed
        self.sim.reset()
        self._dhist.clear()
        self._fc = 1.0
        self._last = {"A": (1.0, 1.0), "B": (1.0, 1.0)}
        oa, ob = self._obs(noisy=True)
        return {"A": oa, "B": ob}

    def step(self, actions):
        pairs = {k: action_to_pair(v) for k, v in actions.items()}
        self._last = {"A": (PRICE_GRID[pairs["A"][0]], INCENTIVE_GRID[pairs["A"][1]]),
                      "B": (PRICE_GRID[pairs["B"][0]], INCENTIVE_GRID[pairs["B"][1]])}
        rec = self.sim.step(pairs["A"], pairs["B"])
        self._dhist.append(rec.demand)
        if self.forecast_fn is not None:
            # forecast the hour ABOUT TO BE simulated (the agent's upcoming action
            # affects it): history runs through hour t-1, target is hour t;
            # rain persistence of the last realized hour is passed as a feature
            t = self.sim.st["t"]
            self._fc = float(self.forecast_fn(list(self._dhist), t % 24, t // 24,
                                              1.0 if self.sim.st["rain"] else 0.0))
        oa, ob = self._obs(noisy=True)
        rewards = {"A": (rec.profit_a - rec.routine_delay_cost_a) / self.cfg.reward_scale,
                   "B": (rec.profit_b - rec.routine_delay_cost_b) / self.cfg.reward_scale}
        info = {"record": rec, "actions_pair": pairs}
        return {"A": oa, "B": ob}, rewards, False, (self.sim.st["t"] >= self.cfg.T), info

    def _obs(self, noisy):
        cfg, st, sim = self.cfg, self.sim.st, self.sim
        rec = sim.records[-1] if sim.records else None
        hour, t = st["t"] % 24, st["t"]
        cap_ref = cfg.n_couriers * cfg.orders_per_courier_hour * cfg.base_participation
        def one(side):
            own_p, own_w = self._last[side]
            comp_p, comp_w = self._last["B" if side == "A" else "A"]
            if rec is None:
                cap = cap_ref
            else:
                cap = (rec.cap_a if side == "A" else rec.cap_b)
                if noisy:
                    cap *= 1.0 + sim.rng.normal(0.0, cfg.obs_noise_std)
            vec = [
                np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24),
                (t // 24) / 6.0, 1.0 if st["rain"] else 0.0,
                st["ewma_demand"] / cfg.mean_lambda,
                (own_p - 1.0) / 0.2, (own_w - 1.0) / 1.0,
                (comp_p - 1.0) / 0.2, (comp_w - 1.0) / 1.0,
                cap / cap_ref,
                st["ewma_ful_a"] if side == "A" else st["ewma_ful_b"],
                (st["ewma_earn_a"] if side == "A" else st["ewma_earn_b"]) / cfg.earn_ref - 1.0,
                (st["ewma_wait_a"] if side == "A" else st["ewma_wait_b"]) / cfg.sla_min,
                st["ewma_can_a"] if side == "A" else st["ewma_can_b"]]
            if self.forecast_fn is not None:
                vec.append(self._fc)
            return np.array(vec, dtype=np.float64)
        return one("A"), one("B")

# ---------- baseline policies (only public information) ----------
class FixedPolicy:
    def __init__(self, price=1.0, incentive=1.0):
        self.pair = [min(range(len(PRICE_GRID)), key=lambda i: abs(PRICE_GRID[i] - price)),
                     min(range(len(INCENTIVE_GRID)), key=lambda i: abs(INCENTIVE_GRID[i] - incentive))]
    def act(self, obs, hint=None):
        return pair_to_action(self.pair)

class RandomPolicy:
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
    def act(self, obs, hint=None):
        return int(self.rng.integers(N_ACTIONS))

class RuleBasedPolicy:
    def __init__(self, price=1.0, incentive=1.0):
        self.pair = [min(range(len(PRICE_GRID)), key=lambda i: abs(PRICE_GRID[i] - price)),
                     min(range(len(INCENTIVE_GRID)), key=lambda i: abs(INCENTIVE_GRID[i] - incentive))]
    def act(self, obs, hint=None):
        cap_ratio, demand, comp_norm = obs[9], obs[4], obs[7]
        comp_idx = int(round(comp_norm * 2)) + 2
        if cap_ratio < 0.90:                      # scarce couriers: price up, incentive up
            self.pair[0] = min(self.pair[0] + 1, 4); self.pair[1] = min(self.pair[1] + 1, 4)
        elif cap_ratio > 1.15 and comp_idx < self.pair[0]:
            self.pair[0] = max(comp_idx, 0)       # match cheaper competitor
            self.pair[1] = max(self.pair[1] - 1, 0)
        elif cap_ratio > 1.15 and demand < 0.85:  # slack + weak demand: discount
            self.pair[0] = max(self.pair[0] - 1, 0); self.pair[1] = max(self.pair[1] - 1, 0)
        return pair_to_action(self.pair)

def run_episode(env, pol_a, pol_b, seed=0):
    obs = env.reset(seed=seed)
    done, tot = False, {"A": 0.0, "B": 0.0}
    while not done:
        obs, r, term, trunc, info = env.step({"A": pol_a.act(obs["A"]), "B": pol_b.act(obs["B"])})
        tot["A"] += r["A"]; tot["B"] += r["B"]
        done = term or trunc
    return tot, info

# ---------- build the calibrated market ----------
CFG = MarketConfig(
    base_lambda=calib_ops["base_lambda"], n_couriers=calib_ops["n_couriers"],
    base_delay_min=calib_ops["base_delay_min"], delay_noise_std=calib_ops["delay_noise_std"],
    sla_min=calib_ops["sla_min"], patience_mean=patience_mean, patience_std=patience_std,
    fixed_cost_per_step=fixed_cost, basket=basket, delivery_fee_ref=fee_ref,
    hour_profile=tuple(hour_profile), day_profile=tuple(day_profile), sigma_lambda=SIGMA_LAMBDA)
print("calibrated market: %.0f orders/hour, %d couriers, delay %.1f+-%.1f min (SLA %.0f), basket %.0f INR, fee %.1f INR"
      % (CFG.base_lambda, CFG.n_couriers, CFG.base_delay_min, CFG.delay_noise_std,
         CFG.sla_min, CFG.basket, CFG.delivery_fee_ref))
env = FoodDeliveryEnv(CFG)
tot, info = run_episode(env, FixedPolicy(1.0, 1.0), FixedPolicy(1.0, 1.0), seed=1)
print("neutral week returns: A=%.0f B=%.0f (reward units)" % (tot["A"], tot["B"]))

In [ ]:
# ===== Metrics: problem-statement operational definitions =====
def gini(values):
    e = np.sort(np.asarray(values, dtype=float)); n = e.size; tot = e.sum()
    if n == 0 or tot <= 0:
        return 0.0
    ranks = np.arange(1, n + 1)
    return float((2.0 * (ranks * e).sum() - (n + 1) * tot) / (n * tot))

def cvar(samples, beta=0.95):
    """Empirical CVaR_beta of tail LOSSES (upper tail = worst outcomes)."""
    x = np.sort(np.asarray(samples, dtype=float))[::-1]
    k = max(int(math.ceil((1.0 - beta) * x.size - 1e-9)), 1)
    return float(x[:k].mean())

def volatility(p_series, w_series):
    dp, dw = np.abs(np.diff(p_series)), np.abs(np.diff(w_series))
    return float(0.5 * dp.mean() + 0.5 * (dw / 2.0).mean()) if dp.size else 0.0

def episode_summary(sim, cfg, spend_benchmark=None):
    """Episode-level aggregates incl. tail loss L = c_f*F + c_s*S + c_w*W."""
    recs = sim.records
    cs = sim.cs
    out = {}
    for side, tag in (("a", "A"), ("b", "B")):
        g = lambda f: np.array([getattr(r, f + "_a") if side == "a" else getattr(r, f + "_b") for r in recs])
        orders = g("fulfilled") + g("unfulfilled")
        fulfilled, cancels, sla = g("fulfilled"), g("cancel"), g("sla")
        profit, spend = g("profit"), g("incentive_spend")
        prices, incs, delays = g("p"), g("w"), g("delay")
        F = float((orders - fulfilled + cancels).sum() / max(orders.sum(), 1e-9))
        S = float((sla * fulfilled).sum() / max(fulfilled.sum(), 1e-9))
        W = (max(0.0, spend.mean() / max(spend_benchmark, 1e-9) - cfg.spend_headroom)
             if spend_benchmark else 0.0)
        elig = (cs.z == 0) if side == "a" else (cs.z == 1)
        out[tag] = dict(
            profit_per_hour=float(profit.mean()),
            fulfillment_rate=float(fulfilled.sum() / max(orders.sum(), 1e-9)),
            F=F, S=S, W=W, tail_loss=cfg.c_f * F + cfg.c_s * S + cfg.c_w * W,
            mean_delay_min=float(delays.mean()),
            cancel_rate=float(cancels.sum() / max(fulfilled.sum(), 1e-9)),
            incentive_spend_total=float(spend.sum()),
            price_volatility=volatility(prices, incs),
            mean_price=float(prices.mean()), mean_incentive=float(incs.mean()),
            courier_gini=gini(cs.earnings[elig]) if elig.any() else 0.0)
    return out

def collusion_proxies(recs, nash=None):
    pa = np.array([r.p_a for r in recs]); pb = np.array([r.p_b for r in recs])
    rho = float(np.corrcoef(pa, pb)[0, 1]) if len(pa) > 2 and pa.std() > 1e-12 and pb.std() > 1e-12 else 0.0
    dpa, dpb = np.diff(pa), np.diff(pb)
    pi = float(((dpa > 0) & (dpb > 0)).mean()) if len(dpa) else 0.0
    dev = {}
    if nash is not None:
        dev = {"benchmark_deviation_A": float(np.mean([r.profit_a for r in recs]) - nash["A"]),
               "benchmark_deviation_B": float(np.mean([r.profit_b for r in recs]) - nash["B"])}
    return dict(price_correlation=rho, parallel_increase_freq=pi, **dev)

def compute_nash(cfg, iters=6):
    """Static Bertrand-Nash via best-response iteration on the 25x25 grid."""
    D = cfg.mean_lambda

    def profit_of(side, pa, pb):
        o = analytical_core(cfg, D, False, PRICE_GRID[pa[0]], INCENTIVE_GRID[pa[1]],
                            PRICE_GRID[pb[0]], INCENTIVE_GRID[pb[1]],
                            cfg.base_delay_min, cfg.base_delay_min, cfg.earn_ref, cfg.earn_ref)
        return o["profit_" + side]

    br = lambda fixed, side: max(((pi, wi) for pi in range(5) for wi in range(5)),
                                 key=lambda act: profit_of(side, act, fixed) if side == "a" else profit_of(side, fixed, act))
    pa = pb = (2, 2)
    for _ in range(iters):
        na = br(pb, "a"); nb = br(na, "b")
        if na == pa and nb == pb:
            break
        pa, pb = na, nb
    o = analytical_core(cfg, D, False, PRICE_GRID[pa[0]], INCENTIVE_GRID[pa[1]],
                        PRICE_GRID[pb[0]], INCENTIVE_GRID[pb[1]],
                        cfg.base_delay_min, cfg.base_delay_min, cfg.earn_ref, cfg.earn_ref)
    return {"A": o["profit_a"], "B": o["profit_b"], "action_a": pa, "action_b": pb}

nash = compute_nash(CFG)
SPEND_BENCHMARK = analytical_core(CFG, CFG.mean_lambda, False, PRICE_GRID[nash["action_a"][0]], 1.0,
                                  PRICE_GRID[nash["action_b"][0]], 1.0, CFG.base_delay_min,
                                  CFG.base_delay_min, CFG.earn_ref, CFG.earn_ref)["incentive_spend_a"]
print("Nash benchmark: actions A=%s B=%s, profit/hour A=%.0f B=%.0f, neutral spend/hour=%.0f"
      % (nash["action_a"], nash["action_b"], nash["A"], nash["B"], SPEND_BENCHMARK))
sim = env.sim
s = episode_summary(sim, CFG, spend_benchmark=SPEND_BENCHMARK)
print("neutral week A: fulfill=%.3f F=%.3f S=%.3f gini=%.2f tail=%.2f | B: fulfill=%.3f tail=%.2f"
      % (s["A"]["fulfillment_rate"], s["A"]["F"], s["A"]["S"], s["A"]["courier_gini"],
         s["A"]["tail_loss"], s["B"]["fulfillment_rate"], s["B"]["tail_loss"]))

In [ ]:
# ===== Stage 3-4: actor-critic networks + PPO/MAPPO trainer =====
import torch
import torch.nn as nn

class Actor(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                 nn.Linear(hidden, hidden), nn.Tanh(),
                                 nn.Linear(hidden, n_actions))
    def dist(self, obs):
        return torch.distributions.Categorical(logits=self.net(obs))

class Critic(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.Tanh(),
                                 nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def greedy_actor(actor):
    def act(obs, hint=None):
        with torch.no_grad():
            logits = actor.net(torch.as_tensor(obs, dtype=torch.float32, device=DEVICE))
            return int(logits.argmax().item())
    return act

class TorchPolicy:
    def __init__(self, act_fn):
        self._act = act_fn
    def act(self, obs, hint=None):
        return self._act(obs, hint)

def discount_gae(rew, val, done, gamma=0.97, lam=0.95):
    T = len(rew); adv = np.zeros(T, dtype=np.float64); last = 0.0
    for t in reversed(range(T)):
        nv = 0.0 if done[t] else (val[t + 1] if t + 1 < T else 0.0)
        delta = rew[t] + gamma * nv - val[t]
        last = delta + gamma * lam * (1.0 - done[t]) * last
        adv[t] = last
    return adv, adv + val[:T]

def stability_gate(logs, tag, min_entropy=0.2):
    """Problem-statement criteria: (1) no divergence, (2) plateau over the
    TRAILING 20% of training (<10% relative change across consecutive windows),
    (3) non-degenerate entropy over the trailing window."""
    ret = np.array([l["return_" + tag] for l in logs], dtype=float)
    if ret.size < 10:
        return {"tag": tag, "verdict": "SKIP"}
    k = max(int(math.ceil(0.2 * ret.size)), 6)
    tail = ret[-k:]
    spread = (tail.max() - tail.min()) / max(abs(tail.mean()), 1e-9)
    n = max(tail.size // 3, 2)
    means = [tail[-(i + 1) * n: tail.size if i == 0 else tail.size - i * n].mean()
             for i in range(3)][::-1]
    rel = max(abs(means[i + 1] - means[i]) / max(abs(means[0]), 1e-9) for i in range(2))
    ent = np.mean([l.get("entropy_" + tag, 1.0) for l in logs[-k:]])
    checks = {"no_divergence": bool(np.isfinite(ret).all() and spread < 10.0),
              "plateau(<10%/window)": rel < 0.10,
              "entropy>=%.1f" % min_entropy: bool(ent >= min_entropy)}
    failed = [k2 for k2, ok in checks.items() if not ok]
    return {"tag": tag, "verdict": "FAIL" if failed else "PASS", "failed": failed,
            "tail_return": float(tail.mean()), "spread": spread, "rel_change": rel, "entropy": float(ent)}

def _checkpoint_path(tag):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(tag))
    ckpt_dir = os.environ.get(
        "GT_CHECKPOINT_DIR",
        "/kaggle/working/gt_model_checkpoints"
        if os.path.isdir("/kaggle/working") else "gt_model_checkpoints"
    )
    os.makedirs(ckpt_dir, exist_ok=True)
    return os.path.join(ckpt_dir, safe + ".pt")

def _save_train_checkpoint(path, actors, critics, opts, ep_global, logs, curves,
                           learners, seed, mode, episodes, obs_dim, crit_in):
    ckpt = {
        "version": 2,
        "mode": mode,
        "seed": int(seed),
        "episodes_target": int(episodes),
        "episode": int(ep_global),
        "obs_dim": int(obs_dim),
        "critic_input_dim": int(crit_in),
        "learners": list(learners),
        "actors": {a: actors[a].state_dict() for a in learners},
        "critics": {a: critics[a].state_dict() for a in learners},
        "optimizers": {a: opts[a].state_dict() for a in learners},
        "logs": logs,
        "curves": curves,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else None
        ),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    tmp = path + ".tmp"
    torch.save(ckpt, tmp)
    os.replace(tmp, path)

def _load_train_checkpoint(path, actors, critics, opts, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    for a in ckpt["learners"]:
        actors[a].load_state_dict(ckpt["actors"][a])
        critics[a].load_state_dict(ckpt["critics"][a])
        opts[a].load_state_dict(ckpt["optimizers"][a])
    if "torch_rng_state" in ckpt:
        torch.set_rng_state(ckpt["torch_rng_state"])
    if "numpy_rng_state" in ckpt:
        np.random.set_state(ckpt["numpy_rng_state"])
    if ckpt.get("cuda_rng_state_all") is not None and torch.cuda.is_available():
        try:
            torch.cuda.set_rng_state_all(ckpt["cuda_rng_state_all"])
        except Exception as exc:
            print("WARNING: could not restore CUDA RNG state:", repr(exc))
    return ckpt

print('Actor / Critic / TorchPolicy / greedy_actor / stability_gate / checkpoint helpers defined')
print('NOTE: the original train() is intentionally NOT defined here - it contained')
print('      the adv_t = zscore(adv_cf) bug. Use train_v2() from FIX-2.')


## 2 — Locate and verify the Model-3 final checkpoint

In [ ]:
# ============================================================
# MODEL 3 CHECKPOINT DISCOVERY
# ============================================================

def _candidate_model3_paths():
    patterns = [
        "/kaggle/input/**/gt_model3_final/final_model.pt",
        "/kaggle/input/**/gt_model3*/**/final_model.pt",
        "/kaggle/working/**/gt_model3_final/final_model.pt",
        "/kaggle/working/**/final_model.pt",
        "gt_model3_final/final_model.pt",
    ]
    out=[]
    for pat in patterns:
        out.extend(glob.glob(pat, recursive=True))
    # Prefer explicit Model-3 paths; never choose a Model-2 counterfactual checkpoint.
    uniq=[]
    seen=set()
    for p in out:
        if os.path.isfile(p) and p not in seen:
            seen.add(p); uniq.append(p)
    uniq.sort(key=lambda p: (0 if "gt_model3" in p.lower() else 1, len(p)))
    return uniq

MODEL3_INIT = None
_candidates = _candidate_model3_paths()
if _candidates:
    MODEL3_INIT = _candidates[0]

print("Model-3 candidate paths:")
for p in _candidates[:20]:
    print("  ", p)

if MODEL3_INIT is None:
    raise FileNotFoundError(
        "Model-3 final_model.pt was not found automatically under /kaggle/input or "
        "/kaggle/working. The Model-4 experiment must start from the validated "
        "Model-3 checkpoint; no Model-2 CF checkpoint will be used as a fallback."
    )

print("\nSELECTED MODEL-3 CHECKPOINT:", MODEL3_INIT)


In [ ]:
# ============================================================
# ROBUST MODEL-3 CHECKPOINT INSPECTION / ACTOR EXTRACTION
# ============================================================
import torch
import torch.nn.functional as F
import torch.nn as nn

DEVICE_M4 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(M4_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(M4_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

print("torch:", torch.__version__, "| device:", DEVICE_M4)

M3_RAW = torch.load(MODEL3_INIT, map_location="cpu", weights_only=False)
print("Model-3 checkpoint type:", type(M3_RAW).__name__)
if isinstance(M3_RAW, dict):
    print("top-level keys:", list(M3_RAW.keys())[:40])

# Actor architecture must match the validated MAPPO actor used in the project.
def _looks_like_actor_state(x):
    if not isinstance(x, dict):
        return False
    keys=set(x.keys())
    required={"net.0.weight","net.0.bias","net.2.weight","net.2.bias","net.4.weight","net.4.bias"}
    if not required.issubset(keys):
        return False
    return tuple(x["net.0.weight"].shape) == (128, 14) and tuple(x["net.4.weight"].shape) == (25, 128)

def _recursive_states(obj, path="root"):
    found=[]
    if _looks_like_actor_state(obj):
        found.append((path, obj))
        return found
    if isinstance(obj, dict):
        for k,v in obj.items():
            found.extend(_recursive_states(v, f"{path}.{k}"))
    elif isinstance(obj, (list,tuple)):
        for i,v in enumerate(obj):
            found.extend(_recursive_states(v, f"{path}[{i}]"))
    return found

_actor_states = _recursive_states(M3_RAW)
print("Actor state-dicts discovered:", len(_actor_states))
for p,_ in _actor_states[:10]:
    print("  ", p)

if len(_actor_states) < 2:
    raise RuntimeError(
        "Could not recover both Model-3 actor state_dicts from final_model.pt. "
        "The checkpoint format is different from the expected MAPPO bundle. "
        "No silent Model-2 fallback is allowed for Model 4."
    )

# Prefer paths mentioning agent A/B. Otherwise preserve discovery order.
state_A = None
state_B = None
for p,s in _actor_states:
    pl=p.lower()
    if state_A is None and (pl.endswith(".a") or ".a." in pl or "['a']" in pl or "\"a\"" in pl or pl.endswith("[a]")):
        state_A=s
    if state_B is None and (pl.endswith(".b") or ".b." in pl or "['b']" in pl or "\"b\"" in pl or pl.endswith("[b]")):
        state_B=s
if state_A is None or state_B is None:
    state_A = _actor_states[0][1]
    state_B = _actor_states[1][1]

print("Model-3 actor A/B states selected successfully.")


In [ ]:
# ============================================================
# MODEL-3 WARM-START ACTORS + MODEL-4 TRAINABLE CRITICS
# ============================================================

class QuantileCritic(nn.Module):
    def __init__(self, in_dim, n_quantiles=25, hidden=128):
        super().__init__()
        self.n_quantiles = int(n_quantiles)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, self.n_quantiles)
        )
        taus = (torch.arange(self.n_quantiles, dtype=torch.float32) + 0.5) / self.n_quantiles
        self.register_buffer("taus", taus)

    def forward(self, x):
        return self.net(x)


def quantile_huber_loss(pred_quantiles, target, taus, kappa=1.0):
    if target.dim() == 1:
        target = target.unsqueeze(1)
    td = target - pred_quantiles
    abs_td = torch.abs(td)
    huber = torch.where(
        abs_td <= kappa,
        0.5 * td.pow(2),
        kappa * (abs_td - 0.5 * kappa)
    )
    indicator = (td.detach() < 0).float()
    return (torch.abs(taus.view(1, -1) - indicator) * huber).mean()


def cvar_from_quantiles(q, alpha=0.10):
    n=q.size(-1)
    k=max(1,int(np.ceil(alpha*n)))
    return torch.sort(q,dim=-1).values[:,:k].mean(dim=-1)


def extract_state_dict(obj, preferred_tokens=()):
    candidates=[]
    def walk(x, path="root"):
        if isinstance(x, dict):
            keys=set(x.keys())
            if keys and all(isinstance(k,str) for k in keys):
                # Quantile critic state dict: same MLP prefix plus final quantile width.
                if "net.0.weight" in keys and "net.4.weight" in keys:
                    candidates.append((path,x))
            for k,v in x.items():
                walk(v, f"{path}.{k}")
        elif isinstance(x,(list,tuple)):
            for i,v in enumerate(x): walk(v,f"{path}[{i}]")
    walk(obj)
    for p,s in candidates:
        pl=p.lower()
        if any(t in pl for t in preferred_tokens): return s
    return candidates[0][1] if candidates else None

actors={"A":Actor(14,N_ACTIONS).to(DEVICE_M4), "B":Actor(14,N_ACTIONS).to(DEVICE_M4)}
actors["A"].load_state_dict(state_A, strict=True)
actors["B"].load_state_dict(state_B, strict=True)

crit_input_dim=28
qcritics={a:QuantileCritic(crit_input_dim,M4_NUM_QUANTILES).to(DEVICE_M4) for a in ("A","B")}
risk_critics={a:Critic(crit_input_dim).to(DEVICE_M4) for a in ("A","B")}

# Try to warm-start quantile critics too, but NEVER require it: Model 4 is allowed
# to retrain the distributional/risk critic heads while preserving the validated actor.
for agent in ("A","B"):
    st = extract_state_dict(M3_RAW, preferred_tokens=(agent.lower(), "quant", "critic"))
    if st is not None:
        try:
            missing, unexpected = qcritics[agent].load_state_dict(st, strict=False)
            print(f"quantile critic {agent}: warm-start attempted | missing={len(missing)} unexpected={len(unexpected)}")
        except Exception as exc:
            print(f"quantile critic {agent}: not loaded ({type(exc).__name__}) -> fresh critic retained")

for m in actors.values(): m.train()
for m in qcritics.values(): m.train()
for m in risk_critics.values(): m.train()

# Keep an immutable CPU copy of Model-3 actor states for all ablations.
M3_ACTOR_REFERENCE = {a:{k:v.detach().cpu().clone() for k,v in actors[a].state_dict().items()} for a in ("A","B")}

print("Model-3 actor warm-start verified.")
print("A actor params:", sum(p.numel() for p in actors["A"].parameters()))
print("B actor params:", sum(p.numel() for p in actors["B"].parameters()))


## 3 — Adaptive market-risk estimator

In [ ]:
# ============================================================
# ADAPTIVE RISK ESTIMATOR
# ============================================================

class AdaptiveRiskEstimator:
    """Maps observable market stress signals to a bounded [0,1] risk score."""
    def __init__(self):
        self.weights=np.array([0.35,0.25,0.25,0.15],dtype=np.float64)

    @staticmethod
    def _clip01(x):
        return float(np.clip(x,0.0,1.0))

    def score(self, obs, demand_history, prev_comp_norm):
        # 1) Demand uncertainty: coefficient of variation over the recent demand history.
        h=np.asarray(demand_history,dtype=float)
        if h.size >= 4:
            mu=max(float(h.mean()),1e-6)
            demand_unc=float(h.std(ddof=0)/mu)
        else:
            demand_unc=0.0
        demand_unc=self._clip01(demand_unc/0.75)

        # 2) Courier scarcity: lower capacity ratio = higher risk.
        scarcity=self._clip01((1.0-float(obs[9]))/0.40)

        # 3) SLA risk: EWMA waiting time above 1x SLA.
        sla_ratio=max(float(obs[12]),0.0)
        sla_risk=self._clip01(max(0.0,sla_ratio-0.85)/0.65)

        # 4) Competitor volatility: normalized movement of competitor price.
        comp=float(obs[7])
        comp_vol=0.0 if prev_comp_norm is None else abs(comp-prev_comp_norm)/2.0
        comp_vol=self._clip01(comp_vol)

        x=np.array([demand_unc,scarcity,sla_risk,comp_vol],dtype=np.float64)
        score=float(np.dot(self.weights,x))
        return self._clip01(score), {
            "demand_uncertainty":float(x[0]),
            "courier_scarcity":float(x[1]),
            "sla_risk":float(x[2]),
            "competitor_volatility":float(x[3]),
        }

    def lambda_t(self, risk_score):
        # Smoothly move around the Model-3 base risk coefficient.
        mult=0.50 + M4_RISK_SCALE*float(np.clip(risk_score,0,1))
        return float(M4_BASE_RISK_COEF*mult)

risk_estimator=AdaptiveRiskEstimator()

# Deterministic unit checks.
_dummy_obs=np.zeros(14,dtype=np.float64); _dummy_obs[9]=1.0; _dummy_obs[12]=0.50; _dummy_obs[7]=0.0
r0,_=risk_estimator.score(_dummy_obs,[60,60,60,60],0.0)
r1,_=risk_estimator.score(_dummy_obs,[20,120,15,130],2.0)
print("risk smoke test -> normal %.3f / stressed %.3f" % (r0,r1))
assert 0.0 <= r0 <= 1.0 and 0.0 <= r1 <= 1.0 and r1 > r0
print("adaptive risk estimator: PASS")


## 4 — RAG knowledge base + deterministic retrieval

In [ ]:
# ============================================================
# RAG — PROJECT-SPECIFIC KNOWLEDGE BASE
# ============================================================

RAG_DOCUMENTS=[
    ("doc_01","Normal market","When demand, courier capacity, SLA performance and competitor prices are stable, preserve competitive pricing and avoid excessive risk aversion."),
    ("doc_02","Demand shock","High demand uncertainty increases the probability of fulfillment pressure. Prefer robust actions and avoid aggressive policies that depend on precise demand estimates."),
    ("doc_03","Courier scarcity","When courier capacity is scarce, higher courier incentives can protect fulfillment, while price adjustments may help moderate incoming demand."),
    ("doc_04","SLA pressure","When waiting time approaches the SLA limit, prioritize actions that reduce operational stress and protect fulfillment over short-term profit."),
    ("doc_05","Competitor volatility","When competitor pricing changes rapidly, avoid overreacting to a single movement and maintain stable decisions unless operational conditions also deteriorate."),
    ("doc_06","Rain and disruption","Weather disruption can increase operational uncertainty. Favor conservative risk-aware choices when delivery capacity is simultaneously constrained."),
    ("doc_07","Historical Model-3 finding","Model 3 improved tail-loss CVaR while keeping competitive profit and improving fulfillment relative to the healthy Model-2 MAPPO reference."),
    ("doc_08","Risk principle","Risk sensitivity should increase under dangerous market conditions and decrease under normal conditions."),
    ("doc_09","RL authority","The learned MAPPO policy remains the final decision-maker. Contextual language guidance is a prior, not direct platform control."),
    ("doc_10","Uncertainty principle","LLM guidance should influence the RL policy more when the contextual recommendation is reliable and less when it is uncertain."),
    ("doc_11","Demand context","Demand forecasts and recent demand patterns are useful contextual variables when selecting price and courier incentive actions."),
    ("doc_12","Historical behavior","Competitor behavior should be interpreted together with demand and capacity conditions instead of being treated as an isolated signal."),
]

KB_TEXT=[x[2] for x in RAG_DOCUMENTS]

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception:
    subprocess.run([sys.executable,"-m","pip","install","-q","scikit-learn"],check=True)
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

RAG_VECTOR= TfidfVectorizer(ngram_range=(1,2),stop_words="english")
RAG_MATRIX=RAG_VECTOR.fit_transform(KB_TEXT)

def rag_retrieve(query, top_k=3):
    q=RAG_VECTOR.transform([query])
    sims=cosine_similarity(q,RAG_MATRIX)[0]
    idx=np.argsort(sims)[::-1][:top_k]
    hits=[]
    for j in idx:
        hits.append({"id":RAG_DOCUMENTS[j][0],"title":RAG_DOCUMENTS[j][1],"text":RAG_DOCUMENTS[j][2],"score":float(sims[j])})
    return hits

rag_test=rag_retrieve("courier shortage, high demand uncertainty and SLA pressure",top_k=3)
print("Top RAG results:")
for h in rag_test:
    print(" ",h["id"],"score=%.3f"%h["score"],"|",h["title"])
assert len(rag_test)==3

with open(M4_ART/"rag_knowledge_base.json","w") as f:
    json.dump({"documents":RAG_DOCUMENTS},f,indent=2)
print("RAG artifact saved ->",M4_ART/"rag_knowledge_base.json")


## 5 — Frozen local LLM policy-prior engine

In [ ]:
# ============================================================
# FROZEN LOCAL LLM BACKEND
# ============================================================
# The RL network is trained; the LLM is frozen. The LLM only produces a
# structured policy prior. It never directly executes an action.
# ============================================================

try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
except Exception:
    subprocess.run([sys.executable,"-m","pip","install","-q","transformers","sentencepiece"],check=True)
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

HF_CACHE=str(M4_ART/"hf_cache")
os.makedirs(HF_CACHE,exist_ok=True)

try:
    LLM_TOKENIZER=AutoTokenizer.from_pretrained(M4_LLM_ID,cache_dir=HF_CACHE)
    LLM_MODEL=AutoModelForSeq2SeqLM.from_pretrained(M4_LLM_ID,cache_dir=HF_CACHE)
except Exception as exc:
    raise RuntimeError(
        f"Could not load the local Hugging Face LLM '{M4_LLM_ID}'. "
        "Enable Kaggle internet for the first run so the model can be cached, "
        "then rerun the cell. Model-4 will not silently replace the LLM with a fake stub."
    ) from exc

LLM_DEVICE=DEVICE_M4
LLM_MODEL=LLM_MODEL.to(LLM_DEVICE).eval()
for p in LLM_MODEL.parameters():
    p.requires_grad_(False)

print("LLM backend loaded:",M4_LLM_ID,"on",LLM_DEVICE)


In [ ]:
# ============================================================
# RAG + LLM PRIOR PARSING + UNCERTAINTY GATE
# ============================================================

LLM_CACHE_PATH=M4_ART/"llm_prior_cache.json"
if LLM_CACHE_PATH.exists():
    try:
        LLM_CACHE=json.loads(LLM_CACHE_PATH.read_text())
    except Exception:
        LLM_CACHE={}
else:
    LLM_CACHE={}

LLM_STATS={"calls":0,"cache_hits":0,"parse_fail":0}

ACTION_PAIRS=[action_to_pair(i) for i in range(N_ACTIONS)]

def _action_desc(a):
    pi,wi=action_to_pair(a)
    return f"A{a}: price_index={pi}, incentive_index={wi}, price={PRICE_GRID[pi]:.0f}, incentive={INCENTIVE_GRID[wi]:.0f}"

def _query_from_context(side,obs,risk_score,risk_features,hits):
    ctx=(
        f"agent={side}; demand_level={float(obs[4]):.2f}; courier_capacity_ratio={float(obs[9]):.2f}; "
        f"sla_ratio={float(obs[12]):.2f}; competitor_price_norm={float(obs[7]):.2f}; "
        f"risk_score={risk_score:.2f}; demand_uncertainty={risk_features['demand_uncertainty']:.2f}; "
        f"courier_scarcity={risk_features['courier_scarcity']:.2f}; sla_risk={risk_features['sla_risk']:.2f}; "
        f"competitor_volatility={risk_features['competitor_volatility']:.2f}; rain={int(obs[3]>0.5)}"
    )
    retrieved=" ".join([f"{h['title']}: {h['text']}" for h in hits])
    return ctx+" | retrieved_context="+retrieved

def _parse_llm_output(text):
    txt=text.strip()
    m=re.search(r"price[_ ]*index\s*[:=]\s*(\d+).*?incentive[_ ]*index\s*[:=]\s*(\d+).*?(?:confidence|conf)\s*[:=]\s*([01](?:\.\d+)?)",txt,re.I|re.S)
    if m is None:
        m=re.search(r"P\s*[:=]\s*(\d+).*?I\s*[:=]\s*(\d+).*?C\s*[:=]\s*([01](?:\.\d+)?)",txt,re.I|re.S)
    if m is None:
        return None
    pi=int(np.clip(int(m.group(1)),0,4)); wi=int(np.clip(int(m.group(2)),0,4)); conf=float(np.clip(float(m.group(3)),0,1))
    return pi,wi,conf

def _smooth_prior(action_idx,confidence):
    coords=np.array(ACTION_PAIRS,dtype=np.float64)
    target=np.array(ACTION_PAIRS[action_idx],dtype=np.float64)
    d=((coords-target)**2).sum(axis=1)
    temp=max(0.35,1.40-0.90*float(confidence))
    logits=-d/temp
    p=np.exp(logits-logits.max()); p/=p.sum()
    return p.astype(np.float32)

def llm_policy_prior(side,obs,risk_score,risk_features,force_refresh=False):
    hits=rag_retrieve(_query_from_context(side,obs,risk_score,risk_features,rag_retrieve(" ".join(map(str,obs.tolist())),top_k=3)),top_k=3)
    query=_query_from_context(side,obs,risk_score,risk_features,hits)
    key=hashlib.sha256(query.encode()).hexdigest()
    if key in LLM_CACHE and not force_refresh:
        LLM_STATS["cache_hits"]+=1
        rec=LLM_CACHE[key]
        return np.asarray(rec["prior"],dtype=np.float32),float(rec["gate"]),hits,rec["raw"]

    action_lines="\n".join(_action_desc(a) for a in range(N_ACTIONS))
    prompt=(
        "You are a cautious contextual decision-support model for a food-delivery platform. "
        "Do not execute the action. Recommend one discrete action only from the 25 candidates. "
        "Consider demand, courier capacity, SLA pressure, competitor behavior, weather and retrieved context. "
        "Return exactly: price_index=<0-4>, incentive_index=<0-4>, confidence=<0-1>.\n\n"
        f"Market context: {query}\n\nCandidates:\n{action_lines}"
    )
    toks=LLM_TOKENIZER(prompt,return_tensors="pt",truncation=True,max_length=512).to(LLM_DEVICE)
    with torch.no_grad():
        out=LLM_MODEL.generate(**toks,max_new_tokens=M4_LLM_MAX_TOKENS,do_sample=False,num_beams=1)
    raw=LLM_TOKENIZER.decode(out[0],skip_special_tokens=True)
    parsed=_parse_llm_output(raw)
    LLM_STATS["calls"]+=1
    if parsed is None:
        LLM_STATS["parse_fail"]+=1
        prior=np.ones(N_ACTIONS,dtype=np.float32)/N_ACTIONS
        gate=0.0
    else:
        pi,wi,conf=parsed
        ai=pair_to_action((pi,wi))
        prior=_smooth_prior(ai,conf)
        retrieval_conf=float(max([h["score"] for h in hits] or [0.0]))
        gate=float(np.clip(0.60*conf+0.40*retrieval_conf,0,1))
        if gate < M4_MIN_GATE:
            gate=0.0
    rec={"prior":prior.tolist(),"gate":gate,"raw":raw,"query":query,"retrieval":hits}
    LLM_CACHE[key]=rec
    try:
        LLM_CACHE_PATH.write_text(json.dumps(LLM_CACHE))
    except Exception:
        pass
    return prior,gate,hits,raw

print("LLM/RAG prior engine ready.")


## 6 — Component smoke test + checkpoint

In [ ]:
# ============================================================
# COMPONENT SMOKE TEST / CHECKPOINT
# ============================================================

env_smoke=FoodDeliveryEnv(CFG)
obs_smoke=env_smoke.reset(seed=99123)
agent_obs=obs_smoke["A"]
score,features=risk_estimator.score(agent_obs,list(env_smoke._dhist),None)
prior,gate,hits,raw=llm_policy_prior("A",agent_obs,score,features,force_refresh=False)

print("risk score      :",round(score,4))
print("risk lambda_t   :",round(risk_estimator.lambda_t(score),4))
print("LLM gate        :",round(gate,4))
print("prior sum       :",float(prior.sum()))
print("prior argmax    :",int(prior.argmax()),_action_desc(int(prior.argmax())))
print("LLM raw output  :",raw[:300])
assert np.isfinite(prior).all() and abs(float(prior.sum())-1.0)<1e-5

setup_ckpt={
    "stage":"4.1-4.3_components",
    "seed":M4_SEED,
    "model3_checkpoint":MODEL3_INIT,
    "llm_model_id":M4_LLM_ID,
    "risk_config":{
        "base_risk_coef":M4_BASE_RISK_COEF,"risk_scale":M4_RISK_SCALE,
        "cvar_limit":M4_CVAR_LIMIT,"cvar_alpha":M4_CVAR_ALPHA
    },
    "rag_documents":len(RAG_DOCUMENTS),
    "llm_stats":LLM_STATS.copy(),
    "created_at":time.strftime("%Y-%m-%d %H:%M:%S")
}
torch.save(setup_ckpt,M4_CKPT/"m4_components_verified.pt")
print("checkpoint saved ->",M4_CKPT/"m4_components_verified.pt")


## 7 — Checkpoint helpers

In [ ]:
# ============================================================
# MODEL-4 CHECKPOINT HELPERS
# ============================================================

def _atomic_torch_save(obj,path):
    path=Path(path); tmp=Path(str(path)+".tmp")
    torch.save(obj,tmp)
    os.replace(tmp,path)

def _restore_rng(ckpt):
    if "torch_rng_state" in ckpt: torch.set_rng_state(ckpt["torch_rng_state"])
    if "numpy_rng_state" in ckpt: np.random.set_state(ckpt["numpy_rng_state"])
    if "python_random_state" in ckpt: random.setstate(ckpt["python_random_state"])
    if ckpt.get("cuda_rng_state_all") is not None and torch.cuda.is_available():
        try: torch.cuda.set_rng_state_all(ckpt["cuda_rng_state_all"])
        except Exception as exc: print("CUDA RNG restore warning:",repr(exc))

def save_m4_checkpoint(path, mode, seed, update, actors, qcritics, risk_critics, opts, lambda_cvar, logs, config_extra=None):
    ckpt={
        "format_version":1,
        "model":"Model4_AdaptiveRisk_LLM_RAG_UncertaintyGated_MAPPO",
        "mode":mode,"seed":int(seed),"update":int(update),
        "obs_dim":14,"critic_input_dim":28,"n_actions":25,
        "actors":{a:{k:v.detach().cpu() for k,v in actors[a].state_dict().items()} for a in ("A","B")},
        "quantile_critics":{a:{k:v.detach().cpu() for k,v in qcritics[a].state_dict().items()} for a in ("A","B")},
        "risk_critics":{a:{k:v.detach().cpu() for k,v in risk_critics[a].state_dict().items()} for a in ("A","B")},
        "optimizers":{a:opts[a].state_dict() for a in ("A","B")},
        "lambda_cvar":{a:float(lambda_cvar[a]) for a in ("A","B")},
        "logs":logs,
        "model3_checkpoint":MODEL3_INIT,
        "m4_config":config_extra or {},
        "torch_rng_state":torch.get_rng_state(),
        "numpy_rng_state":np.random.get_state(),
        "python_random_state":random.getstate(),
        "cuda_rng_state_all":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "saved_at":time.strftime("%Y-%m-%d %H:%M:%S")
    }
    _atomic_torch_save(ckpt,path)

def load_m4_checkpoint(path, mode, actors, qcritics, risk_critics, opts):
    ckpt=torch.load(path,map_location=DEVICE_M4,weights_only=False)
    assert ckpt["mode"]==mode, f"Checkpoint mode mismatch: {ckpt['mode']} vs {mode}"
    for a in ("A","B"):
        actors[a].load_state_dict(ckpt["actors"][a])
        qcritics[a].load_state_dict(ckpt["quantile_critics"][a])
        risk_critics[a].load_state_dict(ckpt["risk_critics"][a])
        opts[a].load_state_dict(ckpt["optimizers"][a])
    _restore_rng(ckpt)
    return ckpt

print("checkpoint helpers ready")


In [ ]:
# ============================================================
# TRAINING MATH / RISK FEATURES
# ============================================================

def gae(rewards,values,dones,gamma=M4_GAMMA,lam=M4_GAE_LAM):
    rewards=np.asarray(rewards,dtype=np.float64)
    values=np.asarray(values,dtype=np.float64)
    dones=np.asarray(dones,dtype=np.float64)
    T=len(rewards); adv=np.zeros(T,dtype=np.float64); last=0.0
    for t in reversed(range(T)):
        nv=0.0 if dones[t] else (values[t+1] if t+1<len(values) else 0.0)
        delta=rewards[t]+gamma*nv-values[t]
        last=delta+gamma*lam*(1.0-dones[t])*last
        adv[t]=last
    return adv,adv+values[:T]

def step_risk_cost(rec,side,cfg):
    fulfilled=float(getattr(rec,"fulfilled_"+side))
    unfulfilled=float(getattr(rec,"unfulfilled_"+side))
    canc=float(getattr(rec,"cancel_"+side))
    sla=float(getattr(rec,"sla_"+side))
    cap=max(float(getattr(rec,"cap_"+side)),1e-6)
    orders=max(fulfilled+unfulfilled,1.0)
    fail=(unfulfilled+canc)/orders
    sla_rate=(sla/max(fulfilled,1.0))
    load=max(0.0,orders/cap-1.0)
    return float(fail+0.50*sla_rate+0.25*load)

def llm_augmented_logits(actor,obs_tensor,prior=None,gate=0.0):
    raw=actor.net(obs_tensor)
    if prior is None or gate<=0:
        return raw
    lp=torch.log(torch.as_tensor(np.clip(prior,1e-8,1),dtype=torch.float32,device=obs_tensor.device))
    return raw + float(gate*M4_LLM_PRIOR_COEF)*lp

def _model_config():
    return {
        "seed":M4_SEED,"rollout":M4_ROLLOUT,"lr":M4_LR,"gamma":M4_GAMMA,"gae_lambda":M4_GAE_LAM,
        "ppo_clip":M4_CLIP,"epochs":M4_EPOCHS,"entropy_coef":M4_ENT_COEF,
        "base_risk_coef":M4_BASE_RISK_COEF,"risk_scale":M4_RISK_SCALE,
        "cvar_limit":M4_CVAR_LIMIT,"cvar_alpha":M4_CVAR_ALPHA,
        "llm_id":M4_LLM_ID,"llm_refresh_updates":M4_LLM_REFRESH_UPDATES,
        "llm_prior_coef":M4_LLM_PRIOR_COEF,"kl_coef":M4_KL_COEF
    }


## 8 — Stage 4.4: Adaptive-risk control (no LLM)

In [ ]:
# ============================================================
# GENERIC MODEL-4 TRAINER
# ============================================================

def _fresh_models_from_m3(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    aa={"A":Actor(14,N_ACTIONS).to(DEVICE_M4),"B":Actor(14,N_ACTIONS).to(DEVICE_M4)}
    aa["A"].load_state_dict(M3_ACTOR_REFERENCE["A"]); aa["B"].load_state_dict(M3_ACTOR_REFERENCE["B"])
    qq={a:QuantileCritic(28,M4_NUM_QUANTILES).to(DEVICE_M4) for a in ("A","B")}
    rr={a:Critic(28).to(DEVICE_M4) for a in ("A","B")}
    return aa,qq,rr

def _make_opts(aa,qq,rr):
    return {a:torch.optim.Adam(list(aa[a].parameters())+list(qq[a].parameters())+list(rr[a].parameters()),lr=M4_LR) for a in ("A","B")}

def _rollout_update(env, aa, qq, rr, opts, lambda_cvar, seed, update, use_adaptive=True, use_llm=True, logs=None):
    obs=env.reset(seed=seed*100003+update)
    data={a:{k:[] for k in ["obs","joint","action","old_logp","reward","risk_cost","done","risk_score","gate","prior"]} for a in ("A","B")}
    demand_hist=[]; prev_comp={"A":None,"B":None}
    last_priors={"A":(None,0.0),"B":(None,0.0)}
    total={"A":0.0,"B":0.0}
    lambda_snapshot={"A":[] ,"B":[]}

    for t in range(M4_ROLLOUT):
        joint=np.concatenate([obs["A"],obs["B"]]).astype(np.float32)
        actions={}
        step_meta={}
        for a,b in (("A","B"),("B","A")):
            score,features=risk_estimator.score(obs[a],demand_hist,prev_comp[a])
            lam_t=risk_estimator.lambda_t(score) if use_adaptive else M4_BASE_RISK_COEF
            force=((update % max(M4_LLM_REFRESH_UPDATES,1))==0 and t==0)
            if use_llm:
                # refresh once at the start of selected rollouts and reuse it through the 25-step rollout
                if force or last_priors[a][0] is None:
                    prior,gate,_,_=llm_policy_prior(a,obs[a],score,features,force_refresh=force)
                    last_priors[a]=(prior,gate)
                prior,gate=last_priors[a]
            else:
                prior,gate=None,0.0
            ot=torch.as_tensor(obs[a],dtype=torch.float32,device=DEVICE_M4).unsqueeze(0)
            with torch.no_grad():
                logits=llm_augmented_logits(aa[a],ot,prior,gate)
                dist=torch.distributions.Categorical(logits=logits)
                act=dist.sample()
                old_logp=float(dist.log_prob(act).item())
            actions[a]=int(act.item())
            step_meta[a]=(score,lam_t,gate,prior,features,old_logp)

        nxt,rewards,terminated,truncated,info=env.step(actions)
        rec=info["record"]
        demand_hist.append(float(rec.demand))
        if len(demand_hist)>48: demand_hist=demand_hist[-48:]
        prev_comp["A"]=float(obs["B"][7]); prev_comp["B"]=float(obs["A"][7])
        done=bool(terminated or truncated)

        for a in ("A","B"):
            risk_score,lam_t,gate,prior,features,agent_old_logp=step_meta[a]
            rc=step_risk_cost(rec,"a" if a=="A" else "b",CFG)
            utility=float(rewards[a])-lam_t*rc
            total[a]+=float(rewards[a])
            data[a]["obs"].append(obs[a].copy())
            data[a]["joint"].append(joint.copy())
            data[a]["action"].append(actions[a])
            data[a]["old_logp"].append(float(agent_old_logp))
            data[a]["reward"].append(utility)
            data[a]["risk_cost"].append(rc)
            data[a]["done"].append(done)
            data[a]["risk_score"].append(risk_score)
            data[a]["gate"].append(gate)
            data[a]["prior"].append(np.ones(N_ACTIONS,dtype=np.float32)/N_ACTIONS if prior is None else prior)
            lambda_snapshot[a].append(lam_t)

        obs=nxt
        if done:
            obs=env.reset(seed=seed*100003+update+7919)
            demand_hist=[]; prev_comp={"A":None,"B":None}; last_priors={"A":(None,0.0),"B":(None,0.0)}

    for a in ("A","B"):
        obs_t=torch.as_tensor(np.stack(data[a]["obs"]),dtype=torch.float32,device=DEVICE_M4)
        joint_t=torch.as_tensor(np.stack(data[a]["joint"]),dtype=torch.float32,device=DEVICE_M4)
        acts_t=torch.as_tensor(data[a]["action"],dtype=torch.long,device=DEVICE_M4)
        old_lp_t=torch.as_tensor(data[a]["old_logp"],dtype=torch.float32,device=DEVICE_M4)
        rew=np.asarray(data[a]["reward"],dtype=np.float64)
        risk=np.asarray(data[a]["risk_cost"],dtype=np.float64)
        done=np.asarray(data[a]["done"],dtype=np.float64)
        with torch.no_grad():
            old_v=qq[a](joint_t).mean(dim=-1).cpu().numpy()
            old_risk=rr[a](joint_t).cpu().numpy()
        adv,ret=gae(rew,np.concatenate([old_v,[old_v[-1]]]),done)
        risk_adv,risk_ret=gae(risk,np.concatenate([old_risk,[old_risk[-1]]]),done)
        # Explicit primal-dual risk constraint on the empirical tail-loss metric from the rollout.
        tail_vec=[]
        # The current env contains the latest partial trajectory; use its operational summary.
        try:
            summ=episode_summary(env.sim,CFG,spend_benchmark=SPEND_BENCHMARK)[a]
            empirical_tail=float(summ["tail_loss"])
        except Exception:
            empirical_tail=0.0
        lambda_cvar[a]=max(0.0,float(lambda_cvar[a])+M4_LAMBDA_LR*max(0.0,empirical_tail-M4_CVAR_LIMIT))
        constrained_adv=adv-float(lambda_cvar[a])*risk_adv
        constrained_adv=(constrained_adv-constrained_adv.mean())/(constrained_adv.std()+1e-8)
        data[a]["_tensors"]=(obs_t,joint_t,acts_t,old_lp_t,torch.as_tensor(constrained_adv,dtype=torch.float32,device=DEVICE_M4),torch.as_tensor(ret,dtype=torch.float32,device=DEVICE_M4),torch.as_tensor(risk_ret,dtype=torch.float32,device=DEVICE_M4),torch.as_tensor(np.stack(data[a]["prior"]),dtype=torch.float32,device=DEVICE_M4),torch.as_tensor(data[a]["gate"],dtype=torch.float32,device=DEVICE_M4))

    losses={}
    for epoch in range(M4_EPOCHS):
        for a in ("A","B"):
            obs_t,joint_t,acts_t,old_lp_t,adv_t,ret_t,risk_ret_t,prior_t,gate_t=data[a]["_tensors"]
            logits=llm_augmented_logits(aa[a],obs_t,prior_t.mean(dim=0).detach().cpu().numpy(),float(gate_t.mean().item())) if False else aa[a].net(obs_t)
            # Per-sample prior must be applied separately.
            raw_logits=aa[a].net(obs_t)
            log_prior=torch.log(torch.clamp(prior_t,min=1e-8))
            logits_eff=raw_logits + (M4_LLM_PRIOR_COEF*gate_t.unsqueeze(1))*log_prior if use_llm else raw_logits
            dist=torch.distributions.Categorical(logits=logits_eff)
            new_lp=dist.log_prob(acts_t)
            ratio=torch.exp(new_lp-old_lp_t)
            s1=ratio*adv_t; s2=torch.clamp(ratio,1.0-M4_CLIP,1.0+M4_CLIP)*adv_t
            policy_loss=-torch.min(s1,s2).mean()
            entropy=dist.entropy().mean()
            q=qq[a](joint_t)
            qloss=quantile_huber_loss(q,ret_t,qq[a].taus)
            rv=rr[a](joint_t)
            rloss=F.mse_loss(rv,risk_ret_t)
            if use_llm:
                target_prior=prior_t.detach()
                kl=(F.kl_div(torch.log_softmax(logits_eff,dim=-1),target_prior,reduction="none").sum(dim=-1)*gate_t).mean()
            else:
                kl=torch.zeros((),device=DEVICE_M4)
            loss=policy_loss+M4_VALUE_COEF*qloss+M4_RISK_VALUE_COEF*rloss+M4_KL_COEF*kl-M4_ENT_COEF*entropy
            opts[a].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(aa[a].parameters())+list(qq[a].parameters())+list(rr[a].parameters()),M4_MAX_GRAD_NORM)
            opts[a].step()
            losses[a]={"loss":float(loss.item()),"policy":float(policy_loss.item()),"q":float(qloss.item()),"risk":float(rloss.item()),"entropy":float(entropy.item()),"kl":float(kl.item()),"lambda_cvar":float(lambda_cvar[a])}
    for a in ("A","B"):
        losses[a]["return"]=float(total[a]); losses[a]["mean_risk_score"]=float(np.mean(data[a]["risk_score"])); losses[a]["mean_llm_gate"]=float(np.mean(data[a]["gate"]))
    return {"losses":losses,"returns":total,"lambda_cvar":{k:float(v) for k,v in lambda_cvar.items()},"llm_stats":LLM_STATS.copy()}

def train_m4(mode,updates,seed,checkpoint_name,resume=True,use_adaptive=True,use_llm=True):
    aa,qq,rr=_fresh_models_from_m3(seed); opts=_make_opts(aa,qq,rr); lambdas={"A":M4_LAMBDA_INIT,"B":M4_LAMBDA_INIT}; logs=[]; start=0
    ckpt_path=M4_CKPT/checkpoint_name
    if resume and ckpt_path.exists():
        ck=load_m4_checkpoint(ckpt_path,mode,aa,qq,rr,opts)
        start=int(ck["update"]); lambdas=ck["lambda_cvar"]; logs=ck.get("logs",[])
        print(f"RESUMED {mode} from update {start}/{updates}")
    env=FoodDeliveryEnv(CFG)
    for upd in range(start,updates):
        out=_rollout_update(env,aa,qq,rr,opts,lambdas,seed,upd,use_adaptive=use_adaptive,use_llm=use_llm,logs=logs)
        row={"update":upd+1,"mode":mode,"return_A":out["returns"]["A"],"return_B":out["returns"]["B"],"lambda_A":out["lambda_cvar"]["A"],"lambda_B":out["lambda_cvar"]["B"]}
        for a in ("A","B"):
            row[f"entropy_{a}"]=out["losses"][a]["entropy"]
            row[f"mean_risk_{a}"]=out["losses"][a]["mean_risk_score"]
            row[f"llm_gate_{a}"]=out["losses"][a]["mean_llm_gate"]
            row[f"loss_{a}"]=out["losses"][a]["loss"]
        logs.append(row)
        if (upd+1)%M4_CKPT_INTERVAL==0 or upd+1==updates:
            save_m4_checkpoint(ckpt_path,mode,seed,upd+1,aa,qq,rr,opts,lambdas,logs,config_extra={**_model_config(),"use_adaptive":use_adaptive,"use_llm":use_llm})
            pd.DataFrame(logs).to_csv(M4_LOG/f"{mode}_training.csv",index=False)
            print(f"[{mode}] update {upd+1:4d}/{updates} | A={row['return_A']:.2f} B={row['return_B']:.2f} | lambda={row['lambda_A']:.3f}/{row['lambda_B']:.3f} | gate={row['llm_gate_A']:.2f}/{row['llm_gate_B']:.2f}")
    return {"actors":aa,"qcritics":qq,"risk_critics":rr,"opts":opts,"lambda_cvar":lambdas,"logs":logs,"checkpoint":str(ckpt_path),"mode":mode}

print("Generic Model-4 trainer ready.")


In [ ]:
# ============================================================
# RUN ADAPTIVE-RISK CONTROL: FULL MODEL 4 MINUS LLM/RAG
# ============================================================

CONTROL=train_m4(
    mode="m4_adaptive_control",
    updates=M4_CONTROL_UPDATES,
    seed=M4_SEED,
    checkpoint_name="m4_adaptive_control_seed_700.pt",
    resume=True,
    use_adaptive=True,
    use_llm=False,
)
print("Adaptive-risk control complete ->",CONTROL["checkpoint"])


## 9 — Evaluate Model-3, adaptive-control, and later integrated policy

In [ ]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def _load_actor_bundle_from_run(run):
    return {a:run["actors"][a] for a in ("A","B")}

class M4EvalAgent:
    def __init__(self,actor,side,use_adaptive,use_llm):
        self.actor=actor.eval(); self.side=side; self.use_adaptive=use_adaptive; self.use_llm=use_llm
        self.reset()
    def reset(self):
        self.hist=[]; self.prev_comp=None; self.cached=(None,0.0); self.cached_update=-999
    def act(self,obs,update_key=0):
        score,features=risk_estimator.score(obs,self.hist,self.prev_comp)
        if self.use_llm and (self.cached[0] is None or update_key-self.cached_update>=M4_LLM_REFRESH_UPDATES):
            self.cached=llm_policy_prior(self.side,obs,score,features,force_refresh=False)[:2]
            self.cached_update=update_key
        prior,gate=(self.cached if self.use_llm else (None,0.0))
        ot=torch.as_tensor(obs,dtype=torch.float32,device=DEVICE_M4).unsqueeze(0)
        with torch.no_grad():
            logits=llm_augmented_logits(self.actor,ot,prior,gate)
            return int(torch.argmax(logits,dim=-1).item()),score,gate

def evaluate_m4(name,actors_pair,n_episodes=12,seed0=50000,use_adaptive=True,use_llm=False,cfg_overrides=None):
    cfg_kwargs=dict(CFG.__dict__)
    if cfg_overrides: cfg_kwargs.update(cfg_overrides)
    cfg_eval=MarketConfig(**cfg_kwargs)
    rows=[]
    for ep in range(n_episodes):
        env=FoodDeliveryEnv(cfg_eval)
        obs=env.reset(seed=seed0+ep)
        pa=M4EvalAgent(actors_pair["A"],"A",use_adaptive,use_llm)
        pb=M4EvalAgent(actors_pair["B"],"B",use_adaptive,use_llm)
        pa.reset(); pb.reset()
        done=False; step=0
        while not done:
            aa,_,_=pa.act(obs["A"],step); bb,_,_=pb.act(obs["B"],step)
            nxt,rewards,term,trunc,info=env.step({"A":aa,"B":bb})
            rec=info["record"]
            pa.hist.append(float(rec.demand)); pb.hist.append(float(rec.demand))
            pa.hist=pa.hist[-48:]; pb.hist=pb.hist[-48:]
            pa.prev_comp=float(obs["B"][7]); pb.prev_comp=float(obs["A"][7])
            obs=nxt; step+=1; done=bool(term or trunc)
        summ=episode_summary(env.sim,cfg_eval,spend_benchmark=SPEND_BENCHMARK)
        rows.append({
            "config":name,"episode":ep,"A_profit_per_hour":summ["A"]["profit_per_hour"],
            "B_profit_per_hour":summ["B"]["profit_per_hour"],"A_fulfillment_rate":summ["A"]["fulfillment_rate"],
            "B_fulfillment_rate":summ["B"]["fulfillment_rate"],"A_cvar_sample":summ["A"]["tail_loss"],
            "B_cvar_sample":summ["B"]["tail_loss"],"A_mean_delay_min":summ["A"]["mean_delay_min"],
            "A_courier_gini":summ["A"]["courier_gini"],"A_mean_price":summ["A"]["mean_price"],
            "A_mean_incentive":summ["A"]["mean_incentive"]
        })
    df=pd.DataFrame(rows)
    tailA=cvar(df["A_cvar_sample"].to_numpy(),CFG.cvar_beta)
    agg={"config":name,"A_profit_per_hour":float(df.A_profit_per_hour.mean()),"B_profit_per_hour":float(df.B_profit_per_hour.mean()),
         "A_fulfillment_rate":float(df.A_fulfillment_rate.mean()),"B_fulfillment_rate":float(df.B_fulfillment_rate.mean()),
         "A_tail_loss_CVaR":float(tailA),"A_mean_delay_min":float(df.A_mean_delay_min.mean()),"A_courier_gini":float(df.A_courier_gini.mean()),
         "A_mean_price":float(df.A_mean_price.mean()),"A_mean_incentive":float(df.A_mean_incentive.mean())}
    return agg,df

# Validate the frozen Model-3 actor reference first.
m3_eval_actors={a:Actor(14,N_ACTIONS).to(DEVICE_M4) for a in ("A","B")}
m3_eval_actors["A"].load_state_dict(M3_ACTOR_REFERENCE["A"]); m3_eval_actors["B"].load_state_dict(M3_ACTOR_REFERENCE["B"])

M3_AGG,M3_EP=evaluate_m4("Model 3 frozen reference",m3_eval_actors,M4_EVAL_EPISODES,50000,use_adaptive=True,use_llm=False)
CTRL_AGG,CTRL_EP=evaluate_m4("Model 4 adaptive control",_load_actor_bundle_from_run(CONTROL),M4_EVAL_EPISODES,50000,use_adaptive=True,use_llm=False)

print(pd.DataFrame([M3_AGG,CTRL_AGG]).round(4).to_string(index=False))
pd.DataFrame([M3_AGG,CTRL_AGG]).to_csv(M4_ART/"stage4_control_evaluation.csv",index=False)


## 10 — Stage 4.5–4.7: full Adaptive Risk + RAG + LLM + Uncertainty Gate

In [ ]:
# ============================================================
# RUN FULL INTEGRATED MODEL 4
# ============================================================

FULL=train_m4(
    mode="m4_full_integrated",
    updates=M4_FULL_UPDATES,
    seed=M4_SEED,
    checkpoint_name="m4_full_integrated_seed_700.pt",
    resume=True,
    use_adaptive=True,
    use_llm=True,
)
print("Full Model 4 complete ->",FULL["checkpoint"])


## 11 — Evaluate the integrated Model 4 under normal and shock conditions

In [ ]:
# ============================================================
# NORMAL + SHOCK EVALUATION
# ============================================================

FULL_AGG_N,FULL_EP_N=evaluate_m4("Model 4 full | normal",_load_actor_bundle_from_run(FULL),M4_EVAL_EPISODES,60000,use_adaptive=True,use_llm=True)
CTRL_AGG_N,CTRL_EP_N=evaluate_m4("Model 4 adaptive | normal",_load_actor_bundle_from_run(CONTROL),M4_EVAL_EPISODES,60000,use_adaptive=True,use_llm=False)
M3_AGG_N,M3_EP_N=evaluate_m4("Model 3 frozen | normal",m3_eval_actors,M4_EVAL_EPISODES,60000,use_adaptive=True,use_llm=False)

SHOCK={"sigma_lambda":0.30}
FULL_AGG_S,FULL_EP_S=evaluate_m4("Model 4 full | shock",_load_actor_bundle_from_run(FULL),M4_EVAL_EPISODES,61000,use_adaptive=True,use_llm=True,cfg_overrides=SHOCK)
CTRL_AGG_S,CTRL_EP_S=evaluate_m4("Model 4 adaptive | shock",_load_actor_bundle_from_run(CONTROL),M4_EVAL_EPISODES,61000,use_adaptive=True,use_llm=False,cfg_overrides=SHOCK)
M3_AGG_S,M3_EP_S=evaluate_m4("Model 3 frozen | shock",m3_eval_actors,M4_EVAL_EPISODES,61000,use_adaptive=True,use_llm=False,cfg_overrides=SHOCK)

EVAL_DF=pd.DataFrame([M3_AGG_N,CTRL_AGG_N,FULL_AGG_N,M3_AGG_S,CTRL_AGG_S,FULL_AGG_S])
print(EVAL_DF.round(4).to_string(index=False))
EVAL_DF.to_csv(M4_ART/"model4_normal_shock_evaluation.csv",index=False)

print("LLM stats:",LLM_STATS)


## 12 — Required ablation: Full Model 4 minus Adaptive Risk

This keeps the LLM/RAG/uncertainty-gated prior but replaces dynamic risk sensitivity with the fixed Model-3 risk coefficient.

In [ ]:
# ============================================================
# FULL MODEL 4 MINUS ADAPTIVE RISK
# ============================================================

FIXED_RISK_LLM=train_m4(
    mode="m4_fixed_risk_llm_ablation",
    updates=M4_ABLATION_UPDATES,
    seed=M4_SEED,
    checkpoint_name="m4_fixed_risk_llm_ablation_seed_700.pt",
    resume=True,
    use_adaptive=False,
    use_llm=True,
)
print("Fixed-risk + LLM ablation complete ->",FIXED_RISK_LLM["checkpoint"])


In [ ]:
# ============================================================
# ABLATION EVALUATION
# ============================================================

FIXED_AGG,FIXED_EP=evaluate_m4("Model 4 fixed-risk + LLM | normal",_load_actor_bundle_from_run(FIXED_RISK_LLM),M4_EVAL_EPISODES,62000,use_adaptive=False,use_llm=True)
ABL_DF=pd.DataFrame([M3_AGG_N,CTRL_AGG_N,FIXED_AGG,FULL_AGG_N,M3_AGG_S,CTRL_AGG_S,FULL_AGG_S])
ABL_DF.to_csv(M4_ART/"model4_ablation_results.csv",index=False)
print(ABL_DF.round(4).to_string(index=False))


## 13 — Multi-seed reproducibility runs

In [ ]:
# ============================================================
# OPTIONAL / FINAL MULTI-SEED RUNS
# ============================================================
# These use the exact same architecture and settings. Set FDM_M4_RUN_MULTI_SEED=0
# to skip when a quick single-seed result is needed.
# ============================================================

RUN_MULTI = int(os.environ.get("FDM_M4_RUN_MULTI_SEED", "0"))
MULTI_ROWS=[]

if RUN_MULTI:
    for seed in (701,702):
        # Each seed has its own checkpoint and its own warm-start from the same Model-3 reference.
        run=train_m4(
            mode=f"m4_full_integrated_seed_{seed}",
            updates=M4_FULL_UPDATES,
            seed=seed,
            checkpoint_name=f"m4_full_integrated_seed_{seed}.pt",
            resume=True,
            use_adaptive=True,
            use_llm=True,
        )
        actors_seed=_load_actor_bundle_from_run(run)
        agg_seed,_=evaluate_m4(f"Model 4 seed {seed}",actors_seed,M4_EVAL_EPISODES,63000+seed,use_adaptive=True,use_llm=True)
        agg_seed["seed"]=seed
        MULTI_ROWS.append(agg_seed)
    MULTI_DF=pd.DataFrame(MULTI_ROWS)
    MULTI_DF.to_csv(M4_ART/"model4_multiseed.csv",index=False)
    print(MULTI_DF.round(4).to_string(index=False))
    print("means:\n",MULTI_DF.select_dtypes(include=[np.number]).mean().round(4).to_string())
else:
    print("Multi-seed run skipped by FDM_M4_RUN_MULTI_SEED=0")


## 14 — Final scientific gate + final checkpoint/artifacts

In [ ]:
# ============================================================
# FINAL MODEL-4 GATE
# ============================================================

# Evidence is reported, never forced positive.
# Primary comparison is full Model 4 vs frozen Model 3.
profit_delta_pct=100.0*(FULL_AGG_N["A_profit_per_hour"]-M3_AGG_N["A_profit_per_hour"])/max(abs(M3_AGG_N["A_profit_per_hour"]),1e-9)
ful_delta=FULL_AGG_N["A_fulfillment_rate"]-M3_AGG_N["A_fulfillment_rate"]
risk_delta=FULL_AGG_N["A_tail_loss_CVaR"]-M3_AGG_N["A_tail_loss_CVaR"]
shock_profit_delta_pct=100.0*(FULL_AGG_S["A_profit_per_hour"]-M3_AGG_S["A_profit_per_hour"])/max(abs(M3_AGG_S["A_profit_per_hour"]),1e-9)
shock_risk_delta=FULL_AGG_S["A_tail_loss_CVaR"]-M3_AGG_S["A_tail_loss_CVaR"]

# Adaptive-risk behavior gate: training logs must show a measurable variation in risk score/lambda.
log_df=pd.DataFrame(FULL["logs"])
lambda_var=float(np.std(log_df["lambda_A"])) if not log_df.empty else float("nan")
gate_mean=float(np.mean(log_df["llm_gate_A"])) if not log_df.empty else float("nan")

checks={
    "profit_competitive_normal": bool(FULL_AGG_N["A_profit_per_hour"] >= 0.95*M3_AGG_N["A_profit_per_hour"]),
    "fulfillment_preserved_normal": bool(FULL_AGG_N["A_fulfillment_rate"] >= M3_AGG_N["A_fulfillment_rate"]-0.02),
    "tail_risk_improved_normal": bool(FULL_AGG_N["A_tail_loss_CVaR"] <= M3_AGG_N["A_tail_loss_CVaR"]),
    "shock_profit_competitive": bool(FULL_AGG_S["A_profit_per_hour"] >= 0.95*M3_AGG_S["A_profit_per_hour"]),
    "shock_tail_risk_not_worse": bool(FULL_AGG_S["A_tail_loss_CVaR"] <= 1.05*M3_AGG_S["A_tail_loss_CVaR"]),
    "adaptive_signal_varies": bool(np.isfinite(lambda_var) and lambda_var>1e-6),
    "llm_gate_is_used": bool(np.isfinite(gate_mean) and gate_mean>0.0),
}

status="PASS" if all(checks.values()) else "FAIL_OR_NULL"

print("="*78)
print("MODEL 4 SCIENTIFIC GATE")
print("="*78)
for k,v in checks.items(): print(f"{k:35s}: {v}")
print("normal profit delta (%)               :",round(profit_delta_pct,3))
print("normal fulfillment delta              :",round(ful_delta,4))
print("normal tail-loss CVaR delta           :",round(risk_delta,4))
print("shock profit delta (%)                :",round(shock_profit_delta_pct,3))
print("shock tail-loss CVaR delta            :",round(shock_risk_delta,4))
print("training lambda std                  :",round(lambda_var,6))
print("mean LLM gate                        :",round(gate_mean,6))
print("FINAL MODEL 4 STATUS                 :",status)
print("="*78)

# Final artifact bundle uses the FULL model checkpoint and records all linked evidence.
FINAL_MODEL_PATH=M4_ART/"final_model.pt"
final_bundle={
    "model_name":"Model 4 — Adaptive Risk + LLM/RAG + Uncertainty-Gated MAPPO",
    "status":status,
    "reference_model3":MODEL3_INIT,
    "full_checkpoint":FULL["checkpoint"],
    "control_checkpoint":CONTROL["checkpoint"],
    "fixed_risk_llm_ablation_checkpoint":FIXED_RISK_LLM["checkpoint"],
    "seed":M4_SEED,
    "config":_model_config(),
    "scientific_checks":checks,
    "metrics":{
        "model3_normal":M3_AGG_N,"adaptive_control_normal":CTRL_AGG_N,"fixed_risk_llm_normal":FIXED_AGG,"model4_normal":FULL_AGG_N,
        "model3_shock":M3_AGG_S,"adaptive_control_shock":CTRL_AGG_S,"model4_shock":FULL_AGG_S
    },
    "llm_stats":LLM_STATS.copy(),
    "rag_documents":RAG_DOCUMENTS,
    "created_at":time.strftime("%Y-%m-%d %H:%M:%S")
}

# Save final state dict bundle separately from the resumable training checkpoint.
final_state={
    "format_version":1,"status":status,"model3_checkpoint":MODEL3_INIT,"seed":M4_SEED,
    "actors":{a:{k:v.detach().cpu() for k,v in FULL["actors"][a].state_dict().items()} for a in ("A","B")},
    "quantile_critics":{a:{k:v.detach().cpu() for k,v in FULL["qcritics"][a].state_dict().items()} for a in ("A","B")},
    "risk_critics":{a:{k:v.detach().cpu() for k,v in FULL["risk_critics"][a].state_dict().items()} for a in ("A","B")},
    "lambda_cvar":FULL["lambda_cvar"],
    "llm_model_id":M4_LLM_ID,
    "rag_artifact":str(M4_ART/"rag_knowledge_base.json"),
    "metrics":final_bundle["metrics"],
}
_atomic_torch_save(final_state,FINAL_MODEL_PATH)
(M4_ART/"final_model_manifest.json").write_text(json.dumps(final_bundle,indent=2,default=str))

# Copy final training log and LLM cache into final artifact folder.
pd.DataFrame(FULL["logs"]).to_csv(M4_ART/"final_training_log.csv",index=False)
try:
    shutil.copy2(LLM_CACHE_PATH,M4_ART/"llm_prior_cache.json")
except Exception:
    pass

print("final model saved ->",FINAL_MODEL_PATH)
print("manifest saved    ->",M4_ART/"final_model_manifest.json")
print("Model 4 artifact directory:",M4_ART)


# Model 4 — completion checkpoint

The final artifact is now saved under:

`/kaggle/working/gt_model4_final/artifacts/final_model.pt`

Resumable checkpoints are under:

`/kaggle/working/gt_model4_final/checkpoints/`

Training logs, RAG knowledge, LLM cache, evaluation tables, ablations, and the final manifest are under:

`/kaggle/working/gt_model4_final/artifacts/` and `/kaggle/working/gt_model4_final/logs/`.

**Do not call Model 4 successful unless the printed scientific gate supports it. A `FAIL_OR_NULL` result is retained as the honest experiment outcome and should not be overwritten.**
